# 01. Explore the ten tables

A teaching notebook: read it top to bottom to learn what is in the State Street case data, how the tables fit together, and where the first problems show up. Every number below is computed from the raw files in `data/`. Nothing in `data/` is changed; the only saved results go to `outputs/`. The few thresholds it uses live in `config/exploration.toml`.

**How to run it:** from the repo root, `uv run jupyter lab` and open this file, then *Run All*. It needs about 4 GB of memory and about two minutes (measured: 109 seconds on an Apple Silicon laptop).

> ### What we found so far
>
> *Written from a full run of this notebook on 15 Oct 2026. Section 11.3 prints the same numbers live, so you can check them after any rerun.*
>
> - **The files are intact.** All ten files match the jury's `manifest.csv` (checksum, rows, columns, size): 10 of 10 PASS.
> - **`transaction_id` is not unique.** 1,409,533 rows but 1,407,428 distinct IDs: 2,105 IDs appear exactly twice, always in adjacent rows, and at the same row positions in `pipeline_events`. A naive join between the two tables creates 4,210 extra rows.
> - **Most repeats are harmless copies, a few are real conflicts.** Of the 2,105 pairs: 2,032 are exact copies, 28 differ only because one copy writes `trade_date` as DD/MM/YYYY, and **45 disagree on a fact** (settlement_date 21, price 14, quantity 6, currency 4, settlement_note 3). The conflicts are blanks, one sign flip, currency swaps, settlement dates that move, and prices off by exactly a factor of 100.
> - **Date formats are mixed.** 9,398 `trade_date` values are DD/MM/YYYY; 1,687 `settlement_date` values are blank. Nothing is unreadable once both formats are handled.
> - **The pipeline copies differ in timing.** 1,987 of the 2,105 pipeline pairs have a different `published_timestamp` (20 `booked_timestamp`, 7 `received_timestamp`); source, batch, event time and cutoff never differ.
> - **Our book has 480 holdings the custodian does not report.** Full outer merge: 480,183 keys in both, 480 only in `positions`, 0 only in `comparison_positions`. None of the 480 is zero, and the trade history supports the book's quantity for 477 of them (BOOKED-only), so they look missing from the custodian feed.
> - **Positions point to master records that do not exist.** 195 account_ids (699 position rows) are missing from `accounts`; 15 security_ids (1,118 position rows) are missing from `securities`, and **all 15 are spin-off targets** in `corporate_actions`, which confirms our main spin-off finding from a second table.
> - **The small tables are clean at their grain but not in content.** All 23 spin-off targets are missing from `securities` (15 traded and held: 1,131 trades, 1,118 positions, 0 prices). All 31 mergers lack a target. Every corporate action carries both a ratio and a cash rate, and 87 of 200 have the ex-date after the effective date. The preferred vendor has no price on the snapshot date for 30 securities. 12 held securities are priced in a currency (XYZ) with no FX rate. The USD rate is never exactly 1 (0.990218 to 1.009784).
> - **Each reversal has two REVERSED records.** All 1,402 REVERSED rows point to a BOOKED original, exactly two per original (701), with the original's own sign. 3,511 BOOKED rows are rebooks of a CANCELLED record, and in 3,328 of the 3,336 rebooks with a new quantity the gross amount still fits the *old* quantity. 842 CANCELLED rows point to IDs that do not exist. The evidence does not settle what REVERSED means; four readings are shown side by side.
> - **The trade-only rebuild reproduces most of the book, and shows how the book was built.**
>   - **Raw benchmark:** 98.77% of comparable keys match `positions`. **BOOKED-only (Guide):** 99.66%. **Our no-repair chain** (dedup, conflicts not evaluable, BOOKED-only, trade date): 99.23%.
>   - **The best fit is not a valid treatment.** It is 99.80%, with `transaction_count` matching on every key, and comes from a policy that counts duplicate copies twice and subtracts *every* REVERSED row: a double offset the Guide warns against.
>   - **The book also seems to keep negative quantities as negative** (655, all BUY, although gross_amount fits the absolute value in 654) **and to count blank quantities as zero** (2,819).
>   - **`positions` is a trade-date book:** the settlement-date version matches only 87.70%.
> - **Two methods agree.** DuckDB reproduces the basic pandas counts and all 180 reconstruction numbers, and all 87 scored numbers from our manual results match this notebook.

## Table of contents

0. Summary (above)
1. [Setup](#setup): find the repo, define the helpers
2. [Integrity check](#integrity): are the files exactly what the jury sent?
3. [Peek at each table](#peek): what one row means
4. [Keys and joins](#keys): is `transaction_id` really unique?
5. [The repeated transaction IDs](#repeats): copies, date formats and real conflicts
6. [Positions versus comparison_positions](#positions): our book against the custodian
7. [The four small tables](#small-tables): corporate actions, prices, FX rates, ledger
8. [Statuses and linked records](#statuses): BOOKED, CANCELLED, REVERSED and what the links mean
9. [Dry-run reconstruction](#rebuild): rebuilding positions from trades, step by step
10. [Defect log](#defects): a table to copy into the defect catalog
11. [Verification and cross-check](#verification): DuckDB second method and our manual results

<a id="setup"></a>
## 1. Setup

The notebook lives in `notebooks/`, but the data lives one folder up. The next cell imports the libraries and finds the repo root by walking up from the current folder until it finds a folder that contains both `manifest.csv` and `data/`. This way the notebook works whether Jupyter was started in the repo root or in `notebooks/`.

In [ ]:
import csv
import gzip
import hashlib
import platform
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_columns', 30)      # show all columns of our widest table
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)   # do not cut off long descriptions


def find_repo_root(start=None):
    """Walk up from `start` until we find the folder that holds manifest.csv and data/."""
    folder = Path(start or Path.cwd()).resolve()
    for candidate in [folder, *folder.parents]:   # the folder itself, then its parent, grandparent, ...
        if (candidate / 'manifest.csv').is_file() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError(f'No folder with manifest.csv and data/ above {folder}')


ROOT = find_repo_root()
OUTPUTS = ROOT / 'outputs'
print('Repo root found:', ROOT.name)
print('Python', platform.python_version(), '| pandas', pd.__version__, '| duckdb', duckdb.__version__)

Next we copy three small helpers from `load_data.ipynb` (the loader the jury gave us). A notebook cannot be imported like a Python module, so copying them is the simplest honest option; they are unchanged except that they use our `ROOT`.

- `manifest()` reads `manifest.csv` (the jury's list of files, row counts and checksums).
- `table_path(name)` turns a table name into the path of its `.csv.gz` file.
- `read_table(name)` loads a table **as text**: `dtype=str` keeps every value as a string, and `keep_default_na=False` stops pandas from turning blanks into `NaN`. This matters because IDs like `000123` would lose their leading zeros as numbers, and a blank is evidence we want to see, not hide.

In [ ]:
# Copied from load_data.ipynb (section 2), unchanged apart from using our ROOT.
def manifest(root=ROOT):
    with (Path(root) / 'manifest.csv').open(newline='', encoding='utf-8') as stream:
        return {row['table']: row for row in csv.DictReader(stream)}

def table_path(name, root=ROOT):
    records = manifest(root)
    if name not in records:
        raise ValueError(f'Unknown table {name!r}. Choose from {sorted(records)}')
    return Path(root) / records[name]['file']

def read_table(name, root=ROOT):
    """Return a pandas frame; preserve all values as raw strings, including blanks."""
    return pd.read_csv(table_path(name, root), dtype=str, keep_default_na=False)

The data dictionary (`data_dictionary.csv`) describes every column of every table. The helper below shows the part of it for one table, so that each time we look at a table we also read what its columns are supposed to mean.

In [ ]:
dictionary = pd.read_csv(ROOT / 'data_dictionary.csv', dtype=str, keep_default_na=False)

def describe_columns(table_name):
    """Return the data dictionary rows (column, intended type, description) for one table."""
    rows = dictionary[dictionary['table'] == table_name]
    return rows[['column', 'intended_type', 'description']].reset_index(drop=True)

describe_columns('fx_rates')   # a small example

**Why we need a date helper.** The Dataset Guide says dates come in two formats: `YYYY-MM-DD` (for example `2026-06-03`) and `DD/MM/YYYY` (for example `03/06/2026`). Both mean 3 June 2026, but as text they are different, so a simple `==` would call them different. We also must not let pandas *guess* the format: `03/06/2026` could be read as 6 March in a US locale.

`normalise_date` tries the ISO format first, then `DD/MM/YYYY` for whatever failed, and returns ISO text. A blank stays blank, and anything that fits neither format is kept **unchanged** (so nothing disappears and two odd values can still be compared). `normalise_timestamp` does the same for the date part of a `YYYY-MM-DD hh:mm:ss` timestamp. `date_format_counts` counts how many values are ISO, slash, blank or unreadable, so failures are always reported.

In [ ]:
def normalise_date(raw_dates):
    """Turn YYYY-MM-DD and DD/MM/YYYY text into YYYY-MM-DD text.
    Blanks stay blank; values in neither format are returned unchanged."""
    iso = pd.to_datetime(raw_dates, format='%Y-%m-%d', errors='coerce')      # NaT where it is not ISO
    slash = pd.to_datetime(raw_dates, format='%d/%m/%Y', errors='coerce')    # NaT where it is not DD/MM/YYYY
    parsed = iso.fillna(slash)                                               # take ISO first, slash second
    as_text = parsed.dt.strftime('%Y-%m-%d')                                 # back to text, NaN where unparsed
    return as_text.fillna(raw_dates)                                         # keep the raw value where unparsed


def normalise_timestamp(raw_timestamps):
    """Normalise the first 10 characters (the date part) and keep the time part as it is."""
    date_part = raw_timestamps.str.slice(0, 10)
    time_part = raw_timestamps.str.slice(10)
    return normalise_date(date_part) + time_part


def date_format_counts(raw_dates):
    """Count how many values are ISO, DD/MM/YYYY, blank, or neither."""
    is_iso = pd.to_datetime(raw_dates, format='%Y-%m-%d', errors='coerce').notna()
    is_slash = pd.to_datetime(raw_dates, format='%d/%m/%Y', errors='coerce').notna()
    is_blank = raw_dates == ''
    return {'ISO YYYY-MM-DD': int(is_iso.sum()),
            'slash DD/MM/YYYY': int(is_slash.sum()),
            'blank': int(is_blank.sum()),
            'unreadable': int((~is_iso & ~is_slash & ~is_blank).sum())}


demo = pd.Series(['2026-06-03', '03/06/2026', '', 'not a date'])
pd.DataFrame({'raw': demo, 'normalised': normalise_date(demo)})

**What this tells us.** Both spellings of 3 June 2026 now become the same text, `2026-06-03`. The blank stays blank and the nonsense value is kept as it was, so it cannot silently disappear. We will use this helper every time we compare dates.

<a id="integrity"></a>
## 2. Integrity check

Before analysing anything we check that the files on our disk are exactly the files the jury published. `verify()` is copied unchanged from `load_data.ipynb`. For each of the ten files it compares four things with `manifest.csv`: the SHA-256 checksum (a fingerprint of the file's bytes), the number of rows, the number of columns and the file size. It reads every file, so it takes a little while. A PASS proves the delivery is intact; it says nothing about whether the records inside are correct.

In [ ]:
# Copied from load_data.ipynb (section 2), unchanged.
def verify(root=ROOT):
    """Check transport integrity and physical counts, not data quality."""
    failures = []
    for name, expected in manifest(root).items():
        path = table_path(name, root)
        if not path.is_file():
            failures.append(f'{name}: file missing')
            continue
        sha = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                sha.update(block)
        with gzip.open(path, 'rt', newline='', encoding='utf-8') as stream:
            reader = csv.reader(stream)
            header = next(reader)
            count = sum(1 for _ in reader)
        ok = (sha.hexdigest() == expected['sha256']
              and count == int(expected['rows'])
              and len(header) == int(expected['columns'])
              and path.stat().st_size == int(expected['bytes']))
        print(f'{name}: {count:,} rows; {"PASS" if ok else "FAIL"}', flush=True)
        if not ok:
            failures.append(name)
    if failures:
        raise SystemExit('Release verification failed: ' + ', '.join(failures))
    print('All ten files verified. This does not establish correctness of their records.')

verify()

**What this tells us.** All ten files PASS: they are byte for byte the files the jury published. Any problem we find later is in the data itself, not caused by a damaged download.

Now we load all ten tables into pandas, as text, and keep them in a dictionary called `tables` (and as separate variables with the table names, for convenience). Then we compare the shape of each loaded table (rows and columns) with `manifest.csv`. This is the official check: the manifest comes from the jury, so a FAIL here would mean we loaded something differently from them.

In [ ]:
TABLE_NAMES = ['accounts', 'securities', 'transactions', 'pipeline_events', 'positions',
               'comparison_positions', 'corporate_actions', 'prices', 'fx_rates', 'ledger_nav']

tables = {}
for name in TABLE_NAMES:
    tables[name] = read_table(name)

accounts = tables['accounts']
securities = tables['securities']
transactions = tables['transactions']
pipeline_events = tables['pipeline_events']
positions = tables['positions']
comparison_positions = tables['comparison_positions']
corporate_actions = tables['corporate_actions']
prices = tables['prices']
fx_rates = tables['fx_rates']
ledger_nav = tables['ledger_nav']

official = manifest()
shape_check = pd.DataFrame([
    {'table': name,
     'manifest_rows': int(official[name]['rows']),
     'loaded_rows': len(tables[name]),
     'manifest_columns': int(official[name]['columns']),
     'loaded_columns': tables[name].shape[1]}
    for name in TABLE_NAMES
])
rows_ok = shape_check['manifest_rows'] == shape_check['loaded_rows']
columns_ok = shape_check['manifest_columns'] == shape_check['loaded_columns']
shape_check['result'] = (rows_ok & columns_ok).map({True: 'PASS', False: 'FAIL'})
shape_check

**What this tells us.** Loading with pandas gives exactly the row and column counts in `manifest.csv` for all ten tables (10 of 10 PASS). Reading everything as text lost no rows, so we start from the complete release.

<a id="peek"></a>
## 3. Peek at each table

One table at a time: first what the data dictionary says about its columns, then the first five raw rows exactly as they are in the file. The aim is to know what a single row *means* (its "grain") before we join anything. We start with the master data (accounts, securities), then the trade flow (transactions, pipeline_events), then holdings (positions, comparison_positions), then reference and valuation data.

### 3.1 `accounts`

The list of client accounts: who holds what, in which fund and share class. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('accounts'))
print(f'accounts: {len(accounts):,} rows, {accounts.shape[1]} columns')
accounts.head(5)

**What this tells us.** **One row = one client account**, with its fund (`fund_id`), share class, base currency and custodian. A blank `close_date` means the account is still open, not that data is missing. `fund_id` is the link to the ledger later.

### 3.2 `securities`

The security master: every instrument the platform is supposed to know about. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('securities'))
print(f'securities: {len(securities):,} rows, {securities.shape[1]} columns')
securities.head(5)

**What this tells us.** **One row = one instrument** the platform knows. Equities have blank `coupon_rate`, `maturity_date` and `par_value` (they do not apply); bonds have them filled. `security_id` is the key we join on; CUSIP and ISIN are only descriptive attributes.

### 3.3 `transactions`

Every trade (BUY or SELL) booked during the observation window. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('transactions'))
print(f'transactions: {len(transactions):,} rows, {transactions.shape[1]} columns')
transactions.head(5)

**What this tells us.** **One row = one trade**: an account buys or sells a quantity of a security at a price, on a trade date, to settle on a settlement date. Already in the first five rows one `trade_date` is written as DD/MM/YYYY, which is why we built the date helper.

### 3.4 `pipeline_events`

The timeline of each trade through the platform: when it was received, booked and published. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('pipeline_events'))
print(f'pipeline_events: {len(pipeline_events):,} rows, {pipeline_events.shape[1]} columns')
pipeline_events.head(5)

**What this tells us.** **One row = the journey of one trade through the platform**: event, received, booked, published, and the cutoff it had to meet. It shares `transaction_id` with `transactions`, so it is the place to measure timeliness.

### 3.5 `positions`

Our own book of record: how many units each account holds of each security on the snapshot date. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('positions'))
print(f'positions: {len(positions):,} rows, {positions.shape[1]} columns')
positions.head(5)

**What this tells us.** **One row = how many units one account holds of one security on 10 July 2026**, according to our own book. Negative quantities appear right away; the Dataset Guide says these can be short positions, not automatically errors.

### 3.6 `comparison_positions`

The same holdings as reported by the custodian, an outside source we reconcile against. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('comparison_positions'))
print(f'comparison_positions: {len(comparison_positions):,} rows, {comparison_positions.shape[1]} columns')
comparison_positions.head(5)

**What this tells us.** **One row = the same kind of holding, as reported by the custodian.** It has 480 fewer rows than `positions`, so some holdings exist on only one side. Section 6 finds out which.

### 3.7 `corporate_actions`

Issuer events (splits, mergers, spin-offs, redemptions, dividends) that can change holdings. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('corporate_actions'))
print(f'corporate_actions: {len(corporate_actions):,} rows, {corporate_actions.shape[1]} columns')
corporate_actions.head(5)

**What this tells us.** **One row = one issuer event** (split, merger, spin-off, redemption, cash dividend). In the first rows only the spin-off has a `target_security_id`; the merger has none, which a later check should look at. Note that the table says nothing about which accounts received the event.

### 3.8 `prices`

Prices from several vendors per security and day. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('prices'))
print(f'prices: {len(prices):,} rows, {prices.shape[1]} columns')
prices.head(5)

**What this tells us.** **One row = one vendor's price for one security on one day.** In the first rows one security has three vendor prices per day, ranked by `vendor_priority` (1 = preferred, not proven correct).

### 3.9 `fx_rates`

Exchange rates to USD per currency and day. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('fx_rates'))
print(f'fx_rates: {len(fx_rates):,} rows, {fx_rates.shape[1]} columns')
fx_rates.head(5)

**What this tells us.** **One row = the USD value of one unit of a currency on one day.** Even the USD rate is not exactly 1 in these rows, the open question already noted in the project summary.

### 3.10 `ledger_nav`

The fund ledger: net assets, shares outstanding and NAV per share per fund and share class. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('ledger_nav'))
print(f'ledger_nav: {len(ledger_nav):,} rows, {ledger_nav.shape[1]} columns')
ledger_nav.head(5)

**What this tells us.** **One row = one share class of one fund on 10 July 2026**, with net assets, shares outstanding and NAV per share. This is the final output the whole platform feeds.

<a id="keys"></a>
## 4. Keys and joins

A **key** is a column (or set of columns) that should identify exactly one row. The Dataset Guide says `transaction_id` is the *intended* key of both `transactions` and `pipeline_events`, and that the link between them is "intended one to one; validate first". So we validate it.

The next cell counts, for both tables: the number of rows, the number of distinct IDs, `duplicated().sum()` (how many rows are a second or later copy of an ID already seen), and how many different IDs repeat. If the key were unique, rows would equal distinct IDs and the last two numbers would be zero.

In [ ]:
def id_summary(table, column):
    """Count rows, distinct values and repeats of one ID column."""
    ids = table[column]
    return {'rows': len(ids),
            'distinct IDs': ids.nunique(),
            'extra copies (duplicated().sum())': int(ids.duplicated().sum()),
            'IDs that repeat': ids[ids.duplicated()].nunique()}

key_check = pd.DataFrame({'transactions': id_summary(transactions, 'transaction_id'),
                          'pipeline_events': id_summary(pipeline_events, 'transaction_id')})
key_check

**What this tells us.** Both tables have 1,409,533 rows but only 1,407,428 distinct IDs: 2,105 IDs repeat, each adding exactly one extra row. So `transaction_id` is **not** a unique key, and the two tables have the same number of repeats.

Next we check coverage in **both directions** with `isin`: does every transaction ID appear in `pipeline_events`, and does every pipeline ID appear in `transactions`? An ID found on only one side would be an orphan: a trade with no pipeline history, or a pipeline record for a trade that does not exist.

In [ ]:
txn_found_in_pipeline = transactions['transaction_id'].isin(pipeline_events['transaction_id'])
pipeline_found_in_txn = pipeline_events['transaction_id'].isin(transactions['transaction_id'])

print('transactions rows whose ID is missing from pipeline_events:', (~txn_found_in_pipeline).sum())
print('pipeline_events rows whose ID is missing from transactions:', (~pipeline_found_in_txn).sum())

same_id_sets = set(transactions['transaction_id']) == set(pipeline_events['transaction_id'])
print('Both tables contain exactly the same set of IDs:', same_id_sets)

**What this tells us.** No orphans in either direction: every trade has pipeline history and every pipeline record belongs to a trade. The only problem with the link is the repeats.

**Why repeated keys are dangerous in a join: many-to-many expansion.** When you merge two tables on a key, pandas pairs *every* matching row on the left with *every* matching row on the right. If an ID appears twice on each side, it produces 2 x 2 = 4 rows, not 2. The toy example below uses made-up IDs (`toy_A`, `toy_B`, not from our data): 3 rows on each side become 5 rows after the merge.

In [ ]:
left = pd.DataFrame({'transaction_id': ['toy_A', 'toy_B', 'toy_B'], 'quantity': [100, 50, 50]})
right = pd.DataFrame({'transaction_id': ['toy_A', 'toy_B', 'toy_B'], 'batch_id': ['B1', 'B2', 'B2']})

toy_joined = left.merge(right, on='transaction_id', how='outer')
print('rows before:', len(left), 'and', len(right), '| rows after:', len(toy_joined))
toy_joined

**What this tells us.** `toy_A` (once on each side) gives 1 row; `toy_B` (twice on each side) gives 2 x 2 = 4 rows. That is how a join can quietly create rows that do not exist, and double count quantities in a position rebuild.

Now the real join. We merge only the ID column of each table (to save memory) with `how='outer'`, so nothing is dropped. Before running it we *predict* the row count: for each ID, (copies in transactions) x (copies in pipeline_events), summed over all IDs. If the prediction matches the actual result, we fully understand where the extra rows come from. The Dataset Guide asks us to report row counts before and after every join; this is why.

In [ ]:
copies_in_txn = transactions['transaction_id'].value_counts()        # copies per ID in transactions
copies_in_pipe = pipeline_events['transaction_id'].value_counts()    # copies per ID in pipeline_events
predicted_rows = (copies_in_txn * copies_in_pipe.reindex(copies_in_txn.index, fill_value=0)).sum()

real_joined = transactions[['transaction_id']].merge(
    pipeline_events[['transaction_id']], on='transaction_id', how='outer', indicator=True)

print(f'rows before: transactions {len(transactions):,}, pipeline_events {len(pipeline_events):,}')
print(f'rows after the outer merge: {len(real_joined):,}')
print(f'predicted from copies per ID: {predicted_rows:,}')
print(f'extra rows created by the join: {len(real_joined) - len(transactions):,}')
print(real_joined['_merge'].value_counts().to_string())
del real_joined   # free memory, we only needed the counts

**What this tells us.** The real merge grows from 1,409,533 to 1,413,743 rows (+4,210), exactly as predicted: each of the 2,105 repeated IDs turns 2 rows into 4. No row is `left_only` or `right_only`. Any later join on `transaction_id` must handle the repeats first.

<a id="repeats"></a>
## 5. The repeated transaction IDs

Now we look closely at the transaction IDs that appear more than once. The questions are: how many copies does each have, where are the copies in the file, are the copies identical, and if not, *how* do they differ?

First we keep only the rows whose ID repeats (`duplicated(keep=False)` marks *every* copy, not only the second one) and remember each row's position in the raw file. The row position is lineage only (where the record sat in the file), never a business key.

In [ ]:
repeated_rows = transactions[transactions['transaction_id'].duplicated(keep=False)].copy()
repeated_rows['row_number'] = repeated_rows.index   # position in the raw file (0 = first data row)

copies_per_id = repeated_rows.groupby('transaction_id').size()
print(f'rows with a repeated ID: {len(repeated_rows):,}')
print(f'distinct repeated IDs: {repeated_rows["transaction_id"].nunique():,}')
print('How many copies each repeated ID has (copies -> number of IDs):')
print(copies_per_id.value_counts().to_string())

**What this tells us.** 4,210 rows carry a repeated ID: 2,105 IDs, and every one appears **exactly twice**. There are no triples, which makes the pairwise comparison below complete.

Where do the copies sit in the file? For each repeated ID we compute the gap between the row numbers of its first and last copy. A gap of 1 means the copies are in adjacent rows. Then we check whether `pipeline_events` has its repeated IDs at exactly the same row positions, which would suggest both files were produced by the same step (for example one record sent twice).

In [ ]:
row_gap = repeated_rows.groupby('transaction_id')['row_number'].agg(lambda rows: rows.max() - rows.min())
print('Gap between first and last copy (gap -> number of IDs):')
print(row_gap.value_counts().to_string())

repeated_pipe = pipeline_events[pipeline_events['transaction_id'].duplicated(keep=False)].copy()
repeated_pipe['row_number'] = repeated_pipe.index

same_positions = list(repeated_rows['row_number']) == list(repeated_pipe['row_number'])
same_ids_at_positions = list(repeated_rows['transaction_id']) == list(repeated_pipe['transaction_id'])
print('Repeated rows sit at the same row positions in both tables:', same_positions)
print('...and carry the same IDs at those positions:', same_ids_at_positions)

**What this tells us.** Every pair sits in **adjacent rows** (gap of 1), and `pipeline_events` has its repeats at the very same row positions with the same IDs. This looks like one record emitted twice by the same upstream step and copied into both files, rather than two independent trades that happen to share an ID.

**Exact copies versus differing copies.** A row is an *exact copy* if another row with the same ID has the same value in every column. `duplicated(keep=False)` on all columns marks every row that has an identical twin.

In [ ]:
data_columns = list(transactions.columns)   # the 16 columns of the file (not our row_number)
is_exact_copy = repeated_rows.duplicated(subset=data_columns, keep=False)

exact_copy_rows = int(is_exact_copy.sum())
exact_copy_pairs = repeated_rows.loc[is_exact_copy, 'transaction_id'].nunique()
print(f'rows that are exact full-row copies: {exact_copy_rows:,} ({exact_copy_pairs:,} IDs)')
print(f'rows whose copies differ somewhere: {len(repeated_rows) - exact_copy_rows:,}')

**What this tells us.** 4,064 rows (2,032 pairs) are perfect copies: identical in all 16 columns. The other 146 rows (73 pairs) differ somewhere, and those are the ones worth a closer look.

To compare copies column by column we put them side by side. `groupby(...).cumcount()` numbers the copies of each ID in file order (0 for the first, 1 for the second). Then `copy_a` holds every first copy and `copy_b` every second copy, both indexed by `transaction_id` so they line up. The code also reports the largest number of copies; if any ID had more than two, copies after the second would not be compared here and we would say so.

In [ ]:
repeated_rows['copy_number'] = repeated_rows.groupby('transaction_id').cumcount()   # 0, 1, ... within each ID

largest_group = copies_per_id.max()
print('largest number of copies of one ID:', largest_group)
if largest_group > 2:
    print('Note: some IDs have more than two copies; only the first two are compared below.')

copy_a = repeated_rows[repeated_rows['copy_number'] == 0].set_index('transaction_id')
copy_b = repeated_rows[repeated_rows['copy_number'] == 1].set_index('transaction_id')
copy_b = copy_b.loc[copy_a.index]   # same order as copy_a so each row is the same ID

compare_columns = [column for column in data_columns if column != 'transaction_id']
raw_differs = pd.DataFrame({column: copy_a[column] != copy_b[column] for column in compare_columns})
print(f'pairs compared: {len(copy_a):,}')
print('Pairs whose raw text differs, per column:')
raw_differs.sum()[raw_differs.sum() > 0]

**What this tells us.** As raw text, the 73 differing pairs disagree in `trade_date` (30), `settlement_date` (21), `price` (14), `quantity` (6), `currency` (4) and `settlement_note` (3). We do not yet know which of these are only a different way of writing the same date.

Some of these differences may be the *same date written two ways*. Before normalising, it helps to see how common the two formats are in the **whole** `transactions` table, for each date column (and the date part of `event_timestamp`).

In [ ]:
format_table = pd.DataFrame({
    'trade_date': date_format_counts(transactions['trade_date']),
    'settlement_date': date_format_counts(transactions['settlement_date']),
    'event_timestamp (date part)': date_format_counts(transactions['event_timestamp'].str.slice(0, 10)),
})
format_table

**What this tells us.** In the whole table, 9,398 `trade_date` values use DD/MM/YYYY; `settlement_date` and `event_timestamp` are always ISO. `settlement_date` is blank 1,687 times. No value is unreadable, so our helper handles every date in this table.

Now we normalise the date columns of both copies with our helper and compare again. A pair that differed in the raw text but is identical after normalising differs **only by date format**: the same record written twice in two formats. Whatever still differs is a **real conflict**: the two copies disagree about a fact. A pair can conflict in more than one column, so the per-column counts can add up to more than the number of pairs.

In [ ]:
def normalise_transaction_dates(frame):
    """Return a copy of a transactions frame with its date columns in YYYY-MM-DD form."""
    frame = frame.copy()
    frame['trade_date'] = normalise_date(frame['trade_date'])
    frame['settlement_date'] = normalise_date(frame['settlement_date'])
    frame['event_timestamp'] = normalise_timestamp(frame['event_timestamp'])
    return frame

norm_a = normalise_transaction_dates(copy_a)
norm_b = normalise_transaction_dates(copy_b)
real_differs = pd.DataFrame({column: norm_a[column] != norm_b[column] for column in compare_columns})

pair_has_raw_difference = raw_differs.any(axis=1)
pair_has_real_conflict = real_differs.any(axis=1)
pair_differs_only_by_format = pair_has_raw_difference & ~pair_has_real_conflict

pair_types = pd.Series({
    'exact copies': int((~pair_has_raw_difference).sum()),
    'differ only by date format': int(pair_differs_only_by_format.sum()),
    'real conflict (at least one column)': int(pair_has_real_conflict.sum()),
})
print(pair_types.to_string())
print('total pairs:', pair_types.sum())
print()
print('Real conflicts per column after normalising (pairs can overlap):')
conflicts_per_column = real_differs.sum()
conflicts_per_column

**What this tells us.** After normalising, all 30 `trade_date` differences disappear. The 2,105 pairs split into **2,032 exact copies, 28 that differ only by date format, and 45 real conflicts**. The real conflicts are in `settlement_date` (21), `price` (14), `quantity` (6), `currency` (4) and `settlement_note` (3); `trade_date` and `event_timestamp` never conflict. The per-column counts add to 48, more than 45, so a few pairs conflict in more than one column.

**What kind of conflict is it?** For each conflicting pair and column we put a label on the difference, using simple rules written before looking at the examples:

- **blank vs value**: one copy is empty, the other is not;
- **currency difference**: the `currency` column has two different codes;
- **sign flip**: both are numbers and one is exactly minus the other (for example 4308 and -4308);
- **other change**: anything else (for example two different settlement dates).

In [ ]:
def as_number(text):
    """Convert text to a float, or None if it is not a number."""
    try:
        return float(text)
    except ValueError:
        return None


def conflict_kind(column, value_a, value_b):
    if value_a == '' or value_b == '':
        return 'blank vs value'
    if column == 'currency':
        return 'currency difference'
    number_a, number_b = as_number(value_a), as_number(value_b)
    if number_a is not None and number_b is not None and number_a == -number_b:
        return 'sign flip'
    return 'other change'


conflict_records = []
for column in compare_columns:
    for txn_id in real_differs.index[real_differs[column]]:    # IDs whose copies conflict in this column
        value_a = norm_a.loc[txn_id, column]
        value_b = norm_b.loc[txn_id, column]
        conflict_records.append({'transaction_id': txn_id, 'column': column,
                                 'copy_a': value_a, 'copy_b': value_b,
                                 'kind': conflict_kind(column, value_a, value_b)})
conflicts = pd.DataFrame(conflict_records)

pd.crosstab(conflicts['column'], conflicts['kind'], margins=True, margins_name='total')

**What this tells us.** Of the 48 column conflicts, 25 are a blank in one copy (price 10, settlement_date 7, quantity 5, settlement_note 3), 4 are currency differences, 1 is a sign flip on quantity, and 18 are other changes (14 different settlement dates, 4 different prices). Blanks suggest one copy lost a field; the others mean the copies disagree about the trade itself.

One example per kind of conflict, picked by code: for each kind we take the first transaction ID in sorted order. For each example we print both raw copies (all columns), so you can see the conflict in context.

In [ ]:
examples = conflicts.sort_values(['kind', 'transaction_id']).groupby('kind').head(1)

for _, example in examples.iterrows():
    print(f"Kind: {example['kind']} | column: {example['column']} | "
          f"copy_a: {example['copy_a']!r} | copy_b: {example['copy_b']!r}")
    both_copies = repeated_rows[repeated_rows['transaction_id'] == example['transaction_id']]
    display(both_copies[['row_number'] + data_columns])

**What this tells us.** The examples show each kind in context. The rows are identical except for one field: a missing price, a currency changed (with price and gross amount unchanged, so at most one currency can be right), a settlement date moved to a Saturday before the trade date, and a quantity whose sign flipped on a BUY. In a position rebuild, keeping the wrong copy changes the result, so these pairs cannot be fixed by simply dropping duplicates.

A closer look at the price conflicts labelled "other change" (both copies have a price, but different ones). Dividing one price by the other shows whether the difference is random or a clean multiple, which would point to a scale (unit) problem such as a price quoted in cents instead of units.

In [ ]:
price_changes = conflicts[(conflicts['column'] == 'price') & (conflicts['kind'] == 'other change')].copy()
price_changes['ratio copy_b / copy_a'] = (price_changes['copy_b'].astype(float)
                                          / price_changes['copy_a'].astype(float)).round(4)
price_changes[['column', 'copy_a', 'copy_b', 'ratio copy_b / copy_a']]

**What this tells us.** The four non-blank price conflicts are not random: one copy is exactly **100 times** the other (three times up, once down). That is the signature of a scale error, such as a price in cents versus units. Which copy is right is unresolved, but `gross_amount` (quantity x price) can tell us in the next notebook.

Are the repeated rows special in their status? We compare the status mix of all transactions with the status mix of the repeated rows.

In [ ]:
status_overall = transactions['status'].value_counts()
status_repeated = repeated_rows['status'].value_counts()
pd.DataFrame({'all transactions': status_overall,
              'repeated rows': status_repeated}).fillna(0).astype(int)

**What this tells us.** The repeated rows are 4,206 BOOKED and 4 CANCELLED, and none REVERSED. Almost all of them are BOOKED, so they would enter a BOOKED-only position rebuild twice unless we deduplicate first.

**The copies in `pipeline_events`.** We pair the repeated pipeline rows the same way (first copy against second copy) and count, per column, how many pairs differ. We normalise the date part of every timestamp first, so a format difference is not counted as a real one.

In [ ]:
pipe_columns = [column for column in pipeline_events.columns if column != 'transaction_id']
timestamp_columns = [column for column in pipe_columns if column.endswith('_timestamp')]

repeated_pipe['copy_number'] = repeated_pipe.groupby('transaction_id').cumcount()
pipe_a = repeated_pipe[repeated_pipe['copy_number'] == 0].set_index('transaction_id')
pipe_b = repeated_pipe[repeated_pipe['copy_number'] == 1].set_index('transaction_id')
pipe_b = pipe_b.loc[pipe_a.index]

for column in timestamp_columns:
    pipe_a[column] = normalise_timestamp(pipe_a[column])
    pipe_b[column] = normalise_timestamp(pipe_b[column])

pipe_differs = pd.DataFrame({column: pipe_a[column] != pipe_b[column] for column in pipe_columns})
print(f'pipeline pairs compared: {len(pipe_a):,}')
print(f'pipeline pairs that are identical in every column: {(~pipe_differs.any(axis=1)).sum():,}')
pipe_differs_per_column = pipe_differs.sum()
pipe_differs_per_column

**What this tells us.** Only 118 pipeline pairs are fully identical. The copies differ in `published_timestamp` (1,987 pairs), `booked_timestamp` (20) and `received_timestamp` (7), and never in `source_system`, `batch_id`, `event_timestamp` or `cutoff_timestamp`. So the duplicate came from the same source event and batch, and the copies were then processed (mostly published) at different times.

How large are the `published_timestamp` differences? We convert both copies to real timestamps and look at the gap (second copy minus first copy). Values that cannot be read become missing and are counted, not dropped.

In [ ]:
published_a = pd.to_datetime(pipe_a['published_timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
published_b = pd.to_datetime(pipe_b['published_timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
print('unreadable published timestamps:', published_a.isna().sum() + published_b.isna().sum())

gap = (published_b - published_a)[pipe_differs['published_timestamp']]   # only pairs that differ
print(f'pairs with a different published_timestamp: {len(gap):,}')
gap.describe()

**What this tells us.** The publication gaps are mostly small: the median is minus 1 minute (the second copy was published one minute earlier) and half the gaps lie between minus 5 and plus 6 minutes. The extremes are large: about 15 hours earlier to 16 hours later. Which copy counts as "the" publication time matters for any timeliness KPI.

<a id="positions"></a>
## 6. Positions versus comparison_positions

`positions` is our book of record; `comparison_positions` is what the custodian reports. Reconciliation means comparing them key by key. The Dataset Guide's key is account + security + snapshot date, and it asks for a **full outer** comparison so that a holding present on only one side stays visible (and is never silently treated as zero).

First, the basics of each table: which snapshot dates appear, how many distinct accounts and securities, and whether the key is unique (the "grain check" the guide asks for before comparing).

In [ ]:
position_key = ['account_id', 'security_id', 'as_of_date']

basics = {}
for name, table in [('positions', positions), ('comparison_positions', comparison_positions)]:
    basics[name] = {
        'rows': len(table),
        'as_of_date values': ', '.join(sorted(table['as_of_date'].unique())),
        'distinct account_ids': table['account_id'].nunique(),
        'distinct security_ids': table['security_id'].nunique(),
        'rows repeating a key': int(table.duplicated(subset=position_key).sum()),
    }
pd.DataFrame(basics)

**What this tells us.** Both tables are one snapshot (`2026-07-10`) and cover the same 100,195 accounts and 6,015 securities. Neither has a repeated key, so the grain check passes and a full outer merge cannot create extra rows.

Now the full outer merge with `indicator=True`. The indicator adds a column `_merge` that says, for each key, whether it was found in both tables (`both`), only in positions (`left_only`) or only in the custodian file (`right_only`). We keep only the key and the quantity from each side and rename the quantities so we can tell them apart.

In [ ]:
book = positions[position_key + ['quantity']].rename(columns={'quantity': 'book_quantity'})
custodian = comparison_positions[position_key + ['quantity']].rename(columns={'quantity': 'custodian_quantity'})

position_compare = book.merge(custodian, on=position_key, how='outer', indicator=True)
side_names = {'both': 'in both', 'left_only': 'only in positions', 'right_only': 'only in comparison_positions'}
position_compare['side'] = position_compare['_merge'].map(side_names)

print(f'rows before: positions {len(book):,}, comparison_positions {len(custodian):,}')
print(f'rows after the outer merge: {len(position_compare):,}')
merge_counts = position_compare['side'].value_counts().reindex(list(side_names.values()), fill_value=0)
merge_counts

**What this tells us.** The merge has 480,663 rows, the same as `positions`, so no row was created or lost. 480,183 keys are in both tables, **480 are only in our book**, and none are only at the custodian. Every custodian holding is matched; the gap is entirely on our side.

The rows found **only in positions** are holdings we think we have but the custodian does not report. We convert their quantity to numbers (counting any value that fails) and summarise: how many, smallest, largest, median, mean, and how many are exactly zero. A zero would be a harmless "empty" position; a non-zero one is a real break to investigate.

In [ ]:
book_only = position_compare[position_compare['side'] == 'only in positions'].copy()
book_only['quantity_number'] = pd.to_numeric(book_only['book_quantity'], errors='coerce')

print('quantities that could not be read as numbers:', book_only['quantity_number'].isna().sum())
print('book-only rows with quantity exactly zero:', (book_only['quantity_number'] == 0).sum())
book_only_summary = book_only['quantity_number'].describe()
book_only_summary

**What this tells us.** All 480 quantities are readable numbers, and **none is zero**, so these are not empty positions that the custodian simply leaves out. They range from -12,322 to 17,690 (median 1,231.5, mean about 870.26). These are real breaks to explain; section 9.10 checks them against the trade history.

**Orphan IDs.** Every `account_id` in positions should exist in `accounts`, and every `security_id` should exist in `securities`. First we confirm that the two master tables have unique keys (otherwise "exists" would be ambiguous), then we list the IDs in positions that have no master record, and count how many position rows they touch.

In [ ]:
print('repeated account_ids in accounts:', accounts['account_id'].duplicated().sum())
print('repeated security_ids in securities:', securities['security_id'].duplicated().sum())

position_account_ids = pd.Series(positions['account_id'].unique())
position_security_ids = pd.Series(positions['security_id'].unique())

orphan_account_ids = position_account_ids[~position_account_ids.isin(accounts['account_id'])]
orphan_security_ids = position_security_ids[~position_security_ids.isin(securities['security_id'])]

rows_with_orphan_account = int(positions['account_id'].isin(orphan_account_ids).sum())
rows_with_orphan_security = int(positions['security_id'].isin(orphan_security_ids).sum())

pd.DataFrame({
    'distinct IDs in positions': [len(position_account_ids), len(position_security_ids)],
    'missing from master table': [len(orphan_account_ids), len(orphan_security_ids)],
    'position rows affected': [rows_with_orphan_account, rows_with_orphan_security],
}, index=['account_id (vs accounts)', 'security_id (vs securities)'])

**What this tells us.** The master tables themselves have unique keys. But `positions` refers to **195 account_ids** (699 position rows) that do not exist in `accounts`, and **15 security_ids** (1,118 position rows) that do not exist in `securities`. That is why positions shows 100,195 accounts and 6,015 securities against 100,000 and 6,012 in the master tables.

Where could securities come from that are held but not in the security master? One candidate is a corporate action that *creates* a new security (a spin-off or merger target). The next cell checks how many of the orphan `security_id`s appear as a `target_security_id` in `corporate_actions`, and for which kind of action. This uses a different table as evidence, which is what the Working Guide asks for before explaining a flag.

In [ ]:
orphan_is_target = orphan_security_ids.isin(corporate_actions['target_security_id'])
print(f'orphan security_ids that are a corporate action target: {orphan_is_target.sum()} of {len(orphan_security_ids)}')

target_actions = corporate_actions[corporate_actions['target_security_id'].isin(orphan_security_ids)]
target_actions['action_type'].value_counts()

**What this tells us.** All 15 missing securities are `target_security_id`s of SPINOFF events. This is independent evidence from a second table for our main finding: spin-offs created new securities that were never set up in the security master, yet are held in 1,118 positions.

<a id="small-tables"></a>
## 7. The four small tables

Section 3 showed what one row of `corporate_actions`, `prices`, `fx_rates` and `ledger_nav` means. Here we test each of them the same way:

1. **The intended key** (from the grain table in the Dataset Guide) and whether it is unique.
2. **Date formats**, with `date_format_counts`, so a DD/MM/YYYY value or an unreadable one is counted, not guessed.
3. **Blanks per column.**
4. **How the table should link** to the others according to the Guide's join table, with coverage counted **in both directions** (how many of A are found in B, and how many of B are found in A).

This is exploration: we look and count, we do not design checks yet. Section 7.5 lists what looks suspicious.

First we read the few settings this part of the notebook needs from `config/exploration.toml` (the snapshot date and the quantity tolerance used in section 9), so no threshold is typed into the code. Then we define three small helpers used for every table.

In [ ]:
import tomllib

with (ROOT / 'config' / 'exploration.toml').open('rb') as stream:
    CONFIG = tomllib.load(stream)

AS_OF_DATE = CONFIG['snapshot']['as_of_date']                       # text, YYYY-MM-DD
TOLERANCE = CONFIG['reconstruction']['quantity_match_tolerance']    # absolute units
print('as-of date:', AS_OF_DATE, '| quantity match tolerance:', TOLERANCE)


def key_report(table, key_columns):
    """Rows, distinct keys and rows that repeat a key, for a key of one or more columns."""
    repeats = int(table.duplicated(subset=key_columns).sum())
    return {'intended key': ' + '.join(key_columns), 'rows': len(table),
            'distinct keys': len(table) - repeats, 'rows repeating a key': repeats}


def blanks_per_column(table):
    """Number of empty strings in each column (the tables are read as text)."""
    return (table == '').sum()


def coverage(left_values, right_values, left_name, right_name):
    """Distinct non-blank values of each side, and how many are found on the other side."""
    left = pd.Series(left_values[left_values != ''].unique())
    right = pd.Series(right_values[right_values != ''].unique())
    rows = []
    for name_from, values_from, name_to, values_to in [(left_name, left, right_name, right),
                                                       (right_name, right, left_name, left)]:
        found = int(values_from.isin(values_to).sum())
        rows.append({'direction': f'{name_from} found in {name_to}', 'distinct values': len(values_from),
                     'found': found, 'not found': len(values_from) - found})
    return pd.DataFrame(rows)

**What this tells us.** The settings are loaded: the snapshot date is 10 July 2026 and the quantity tolerance is one millionth of a unit. The helpers count; they never drop a row.

### 7.1 `corporate_actions`

The Guide's grain is "one issuer event", so the intended key is `action_id`. The table has two dates (`ex_date`, `effective_date`). We check the key, the date formats and the blanks.

In [ ]:
display(pd.DataFrame([key_report(corporate_actions, ['action_id'])]))
display(pd.DataFrame({'ex_date': date_format_counts(corporate_actions['ex_date']),
                      'effective_date': date_format_counts(corporate_actions['effective_date'])}))
blanks_per_column(corporate_actions)

**What this tells us.** `action_id` is unique (200 rows, 200 keys). Both dates are always ISO, with no blanks. The only blanks are 177 `target_security_id` values, which is expected for events that do not create a new security.

**Which fields should each event type use?** Before looking at the data, we write down what the Dataset Guide's conventions (section 4) use for each event type. This is our reading of the Guide, fixed before counting:

| action_type | `ratio` | `cash_rate` | `target_security_id` |
|---|---|---|---|
| SPLIT | used (new units per old unit) | not expected (a split pays no cash) | not expected |
| MERGER | used (target units per source unit) | unclear (cash-and-stock mergers exist; the Guide is silent) | used (the merger needs a target) |
| SPINOFF | used (extra target units per source unit) | not expected | used |
| REDEMPTION | unclear (the Guide asks for "sufficient terms") | used (cash treatment) | not expected |
| CASH_DIVIDEND | not expected | used | not expected |

The next cell counts how many rows of each type have each field filled, and then how many are filled where we marked "not expected" or missing where we marked "used".

In [ ]:
GUIDE_FIELD_USE = pd.DataFrame({
    'ratio':              {'SPLIT': 'used', 'MERGER': 'used', 'SPINOFF': 'used',
                           'REDEMPTION': 'unclear', 'CASH_DIVIDEND': 'not expected'},
    'cash_rate':          {'SPLIT': 'not expected', 'MERGER': 'unclear', 'SPINOFF': 'not expected',
                           'REDEMPTION': 'used', 'CASH_DIVIDEND': 'used'},
    'target_security_id': {'SPLIT': 'not expected', 'MERGER': 'used', 'SPINOFF': 'used',
                           'REDEMPTION': 'not expected', 'CASH_DIVIDEND': 'not expected'},
})
event_fields = list(GUIDE_FIELD_USE.columns)

filled_by_type = pd.DataFrame({'rows': corporate_actions['action_type'].value_counts()})
for column in event_fields:
    filled_by_type[f'{column} filled'] = (corporate_actions[column] != '').groupby(corporate_actions['action_type']).sum()
display(filled_by_type)

field_surprises = []
for column in event_fields:
    for action_type, use in GUIDE_FIELD_USE[column].items():
        rows_of_type = corporate_actions[corporate_actions['action_type'] == action_type]
        filled = int((rows_of_type[column] != '').sum())
        if use == 'not expected' and filled > 0:
            field_surprises.append({'action_type': action_type, 'field': column, 'Guide use': use,
                                    'issue': 'filled where not expected', 'rows of type': len(rows_of_type),
                                    'rows affected': filled})
        if use == 'used' and filled < len(rows_of_type):
            field_surprises.append({'action_type': action_type, 'field': column, 'Guide use': use,
                                    'issue': 'missing where used', 'rows of type': len(rows_of_type),
                                    'rows affected': len(rows_of_type) - filled})
field_surprises = pd.DataFrame(field_surprises)
field_surprises

**What this tells us.** Every row has a `ratio` and a `cash_rate`, whatever the event type. So the fields do not tell us which terms apply: all 66 cash dividends carry a ratio, and all 52 splits and 23 spin-offs carry a cash rate, where our reading of the Guide expects none. The other way round, **none of the 31 mergers names a target security**, although a merger ratio means "target units per source unit". Only the 23 spin-offs have a target. Under the Guide's rule ("missing targets ... must be quarantined, not invented") the mergers cannot be applied as they are.

**Do the securities exist?** The Guide's join table says `corporate_actions.security_id` and `target_security_id` should both be validated against `securities.security_id`. We check coverage in both directions, and then follow the targets that are missing: are they traded, held, or priced anyway?

In [ ]:
print('source security_id:')
display(coverage(corporate_actions['security_id'], securities['security_id'], 'corporate_actions', 'securities'))
print('target_security_id (blank targets are left out and counted below):')
display(coverage(corporate_actions['target_security_id'], securities['security_id'], 'corporate_actions targets', 'securities'))
print('rows with a blank target_security_id:', int((corporate_actions['target_security_id'] == '').sum()))

has_target = corporate_actions['target_security_id'] != ''
target_in_master = corporate_actions['target_security_id'].isin(securities['security_id'])
missing_target_actions = corporate_actions[has_target & ~target_in_master]
missing_targets = pd.Series(missing_target_actions['target_security_id'].unique())

traded_targets = missing_targets[missing_targets.isin(transactions['security_id'])]
held_targets = missing_targets[missing_targets.isin(positions['security_id'])]
target_summary = pd.Series({
    'events whose target is missing from securities': len(missing_target_actions),
    '...of which SPINOFF': int((missing_target_actions['action_type'] == 'SPINOFF').sum()),
    'distinct missing targets': len(missing_targets),
    'missing targets that are traded': len(traded_targets),
    'missing targets that are held': len(held_targets),
    'missing targets traded and held': int(traded_targets.isin(held_targets).sum()),
    'transaction rows on missing targets': int(transactions['security_id'].isin(missing_targets).sum()),
    'position rows on missing targets': int(positions['security_id'].isin(missing_targets).sum()),
    'price rows on missing targets': int(prices['security_id'].isin(missing_targets).sum()),
})
target_summary

**What this tells us.** All 200 source securities exist in `securities`, but **none of the 23 targets does**. All 23 are spin-off targets, and 15 of them are already traded and held: 1,131 transaction rows and 1,118 position rows sit on securities the platform has never set up, and none of them has a single price. The other 8 targets are not used yet.

**Event chronology.** An ex-date is normally on or before the date the event takes effect. We normalise both dates with `normalise_date`, convert them to real dates (values that fail stay visible as "not readable") and compare them, per event type.

In [ ]:
ex_day = pd.to_datetime(normalise_date(corporate_actions['ex_date']), format='%Y-%m-%d', errors='coerce')
effective_day = pd.to_datetime(normalise_date(corporate_actions['effective_date']), format='%Y-%m-%d', errors='coerce')

chronology = pd.Series('not readable', index=corporate_actions.index)
chronology[ex_day < effective_day] = 'ex_date before effective_date'
chronology[ex_day == effective_day] = 'same day'
chronology[ex_day > effective_day] = 'ex_date after effective_date'
ex_after_effective = int((chronology == 'ex_date after effective_date').sum())

pd.crosstab(corporate_actions['action_type'], chronology.rename('chronology'), margins=True, margins_name='total')

**What this tells us.** Only 107 events have the ex-date before the effective date. **87 of 200 have the ex-date after the effective date**, spread over every event type, and 6 fall on the same day. Either the two date columns are swapped in many rows or they follow a convention the dictionary does not state. The faculty question on corporate-action dates is still open (open questions in `docs/STATUS.md`).

**Is there any account information?** The Guide says `corporate_actions` records issuer events, "not which accounts received them". We confirm it from the columns: no column name mentions an account. We also count the candidate holders, that is the accounts that hold a source security in `positions`. Candidate is the right word: holding a security on 10 July does not prove the account was entitled on the ex-date.

In [ ]:
print('columns:', list(corporate_actions.columns))
print('columns that mention an account:', [column for column in corporate_actions.columns if 'account' in column])

source_held = positions[positions['security_id'].isin(corporate_actions['security_id'])]
print(f"source securities held in positions: {source_held['security_id'].nunique()} of "
      f"{corporate_actions['security_id'].nunique()} | position rows: {len(source_held):,} | "
      f"candidate accounts: {source_held['account_id'].nunique():,}")

**What this tells us.** No column refers to an account, as the Guide says. The 200 source securities are held in 15,873 position rows by 14,802 accounts on 10 July. These are only *candidate* holders: to say who was entitled we would need holdings on each ex-date, which a trade-only rebuild could estimate but the table itself does not give.

### 7.2 `prices`

The Guide's grain is "security, price date and vendor", so that is the intended key. The Guide's join table also warns: **select date and vendor before joining** positions to prices, because one security has many prices. We check the key, the formats, the blanks, and then how many dates and vendors each security has.

In [ ]:
display(pd.DataFrame([key_report(prices, ['security_id', 'price_date', 'vendor'])]))
display(pd.DataFrame({'price_date': date_format_counts(prices['price_date']),
                      'price_timestamp (date part)': date_format_counts(prices['price_timestamp'].str.slice(0, 10))}))
blanks_per_column(prices)

**What this tells us.** The key `security_id + price_date + vendor` is unique, both date fields are always ISO, and nothing is blank. So the table is clean at the level of its grain; the questions are about coverage.

How many dates and vendors are there, and does every security have a price from every vendor on every date? We also show which `vendor_priority` each vendor has (lower means preferred, not proven more accurate).

In [ ]:
price_dates = normalise_date(prices['price_date'])
n_price_dates = price_dates.nunique()
n_vendors = prices['vendor'].nunique()
print(f'distinct price dates: {n_price_dates} ({price_dates.min()} to {price_dates.max()})')
print(f'distinct vendors: {n_vendors} | distinct securities: {prices["security_id"].nunique():,}')
display(prices.groupby(['vendor', 'vendor_priority']).size().rename('rows').reset_index())

rows_per_security = prices.groupby('security_id').size()
print(f'Rows per security (a full grid is {n_vendors} vendors x {n_price_dates} dates = {n_vendors * n_price_dates}):')
print(rows_per_security.value_counts().sort_index().to_string())

# For the securities with an incomplete grid: which vendor has fewer dates?
incomplete = rows_per_security[rows_per_security < n_vendors * n_price_dates].index
dates_per_vendor = (prices[prices['security_id'].isin(incomplete)]
                    .groupby(['security_id', 'vendor'])['price_date'].nunique().unstack(fill_value=0))
print('Securities with an incomplete grid, by number of dates each vendor covers:')
display(dates_per_vendor.value_counts().rename('securities').reset_index())

# Which (vendor, date) combinations are missing for those securities?
all_dates = set(price_dates)
missing_vendor_dates = []
for (security_id, vendor), dates in (prices[prices['security_id'].isin(incomplete)]
                                     .assign(price_day=price_dates)
                                     .groupby(['security_id', 'vendor'])['price_day']):
    for missing_day in sorted(all_dates - set(dates)):
        missing_vendor_dates.append({'security_id': security_id, 'vendor': vendor, 'missing date': missing_day})
missing_vendor_dates = pd.DataFrame(missing_vendor_dates)
print('Missing vendor prices (vendor, date -> securities):')
missing_vendor_dates.groupby(['vendor', 'missing date']).size().rename('securities').reset_index()

**What this tells us.** There are 30 price dates (the 30 weekdays from 1 June to 10 July) and 3 vendors with fixed priorities (A = 1, B = 2, C = 3). 5,970 securities have the full grid of 90 rows. **30 securities miss one price: VENDOR_A, the preferred vendor, on 10 July, the snapshot date itself.** On the valuation date those 30 securities must fall back to vendor B or C.

**Links.** `prices.security_id` should exist in `securities`, and the Guide's join table links `positions.security_id` to prices. A held security with no price at all cannot be valued. We count coverage both ways and check whether the unpriced held securities are the orphan spin-off targets from section 6.

In [ ]:
display(coverage(prices['security_id'], securities['security_id'], 'prices', 'securities'))
display(coverage(positions['security_id'], prices['security_id'], 'positions', 'prices'))

held_securities = pd.Series(positions['security_id'].unique())
held_without_price = held_securities[~held_securities.isin(prices['security_id'])]
print(f'held securities with no price at all: {len(held_without_price)} '
      f'(position rows: {int(positions["security_id"].isin(held_without_price).sum()):,})')
print(f'...of which orphan security_ids from section 6: {int(held_without_price.isin(orphan_security_ids).sum())}')
print(f'...of which spin-off targets missing from securities: {int(held_without_price.isin(missing_targets).sum())}')

**What this tells us.** Every priced security exists in `securities`, and every priced security is held. **15 held securities have no price at all (1,118 position rows), and they are exactly the 15 orphan spin-off targets** from section 6. The same defect shows up a third time, now from the price side. 12 master securities have no price, but none of them is held or traded, so they do not affect valuation.

**Currency.** Does each price carry the same currency as its security in `securities`? We merge the security's currency onto every price row (many to one; the row count must not change) and compare.

In [ ]:
security_currency = securities[['security_id', 'currency']].rename(columns={'currency': 'security_currency'})
price_vs_security = prices[['security_id', 'currency']].merge(security_currency, on='security_id',
                                                               how='left', validate='many_to_one')
print(f'rows before: {len(prices):,} | rows after: {len(price_vs_security):,}')
print('price rows whose security is not in securities:', int(price_vs_security['security_currency'].isna().sum()))
print('price rows whose currency differs from the security currency:',
      int((price_vs_security['currency'] != price_vs_security['security_currency']).sum()))
print()
print('Price rows by currency:')
display(prices['currency'].value_counts())

# Securities whose master currency has no FX rate at all: are they used?
fx_currencies = set(fx_rates['currency'])
no_fx_securities = securities.loc[~securities['currency'].isin(fx_currencies), 'security_id']
print(f'securities whose currency has no FX rate: {len(no_fx_securities)} '
      f'(currencies: {sorted(securities.loc[~securities["currency"].isin(fx_currencies), "currency"].unique())})')
print(f'...held: {int(no_fx_securities.isin(positions["security_id"]).sum())} '
      f'({int(positions["security_id"].isin(no_fx_securities).sum()):,} position rows) | '
      f'traded: {int(no_fx_securities.isin(transactions["security_id"]).sum())} '
      f'({int(transactions["security_id"].isin(no_fx_securities).sum()):,} transaction rows)')

unpriced_master = securities[~securities['security_id'].isin(prices['security_id'])]
print(f'securities in the master with no price: {len(unpriced_master)} | held: '
      f'{int(unpriced_master["security_id"].isin(positions["security_id"]).sum())} | traded: '
      f'{int(unpriced_master["security_id"].isin(transactions["security_id"]).sum())}')

**What this tells us.** Price and security currencies always agree (0 differences in 539,970 rows; the merge kept every row). But one currency is not real: **XYZ, used by 12 securities (1,080 price rows), and none of the six FX currencies is XYZ**. These 12 securities are held in 898 positions and traded in 2,649 transactions, so they cannot be converted to USD.

### 7.3 `fx_rates`

The grain is "currency and rate date". `rate_to_usd` is the number of US dollars for one unit of the currency. We check the key, formats and blanks, convert the rate to a number (counting failures), and see which currencies and dates are covered.

In [ ]:
display(pd.DataFrame([key_report(fx_rates, ['currency', 'rate_date'])]))
display(pd.DataFrame({'rate_date': date_format_counts(fx_rates['rate_date'])}))
display(blanks_per_column(fx_rates).rename('blanks').to_frame())

fx = fx_rates.assign(rate_day=normalise_date(fx_rates['rate_date']),
                     rate=pd.to_numeric(fx_rates['rate_to_usd'], errors='coerce'))
print('rates that are not readable numbers:', int(fx['rate'].isna().sum()))
fx.groupby('currency').agg(dates=('rate_day', 'nunique'), first=('rate_day', 'min'), last=('rate_day', 'max'),
                           lowest_rate=('rate', 'min'), highest_rate=('rate', 'max'))

**What this tells us.** The FX key is unique, the dates are ISO, and every rate is a readable number. Each of the 6 currencies (CAD, CHF, EUR, GBP, JPY, USD) has a rate on all 30 dates. The ranges look plausible, except that USD itself moves.

**Does every currency in use have a rate?** The Guide's join table links `prices.currency` and `price_date` to `fx_rates.currency` and `rate_date`. For each price row we look up the rate for its currency on its date. For transactions we do the same with `trade_date` (normalised first, because 9,398 trade dates are DD/MM/YYYY). A row with no rate is counted per currency, not dropped. We also compare the set of FX dates with the set of price dates.

In [ ]:
fx_keys = set(zip(fx['currency'], fx['rate_day']))

def rows_without_rate(currencies, days):
    """Per currency: rows, and rows whose (currency, date) has no FX rate."""
    has_rate = pd.Series([pair in fx_keys for pair in zip(currencies, days)], index=currencies.index)
    return pd.DataFrame({'rows': currencies.value_counts(),
                         'rows without a rate': (~has_rate).groupby(currencies).sum()})

print('prices:')
display(rows_without_rate(prices['currency'], price_dates))
print('transactions (rate looked up on the normalised trade_date):')
display(rows_without_rate(transactions['currency'], normalise_date(transactions['trade_date'])))
print('FX dates equal the price dates:', set(fx['rate_day']) == set(price_dates))

**What this tells us.** Every transaction currency has a rate on every trade date, and the FX dates are exactly the price dates. The only gap is the 1,080 XYZ price rows, which have no rate on any date.

**The USD rate.** One US dollar should be worth exactly one US dollar. We look at the USD rows only.

In [ ]:
usd = fx[fx['currency'] == 'USD']
usd_summary = pd.Series({'rows': len(usd), 'rows exactly 1': int((usd['rate'] == 1).sum()),
                         'lowest': usd['rate'].min(), 'highest': usd['rate'].max(),
                         'mean': round(usd['rate'].mean(), 6)})
usd_summary

**What this tells us.** **The USD rate is never exactly 1**: it ranges from 0.990218 to 1.009784 over the 30 dates (mean 0.998621). Taken literally, converting a USD holding to USD would change its value by up to about 1%. Whether this is a planted error or a convention (for example a rate against another reference) is still open; it is one of our faculty questions.

### 7.4 `ledger_nav`

The grain is "fund, share class and snapshot date", so the intended key is `fund_id + share_class + as_of_date`. The Guide links it to the rest of the platform through `accounts` (each account has a `fund_id` and `share_class`), and positions reach a fund through their account. We do **not** test `net_assets / shares_outstanding` yet; here we only check the level of the table and the links.

In [ ]:
display(pd.DataFrame([key_report(ledger_nav, ['fund_id', 'share_class', 'as_of_date'])]))
display(pd.DataFrame({'as_of_date': date_format_counts(ledger_nav['as_of_date'])}))
display(blanks_per_column(ledger_nav).rename('blanks').to_frame())

ledger_level = pd.Series({
    'distinct fund_id': ledger_nav['fund_id'].nunique(),
    'distinct share_class': ledger_nav['share_class'].nunique(),
    'share classes': ', '.join(sorted(ledger_nav['share_class'].unique())),
    'distinct as_of_date': ledger_nav['as_of_date'].nunique(),
    'as_of_date values': ', '.join(sorted(ledger_nav['as_of_date'].unique())),
})
print(ledger_level.to_string())
print('Share classes per fund (classes -> funds):')
print(ledger_nav.groupby('fund_id')['share_class'].nunique().value_counts().to_string())

**What this tells us.** The ledger is unique at the level of `fund_id + share_class + as_of_date`. It has one row for each of 500 funds x 4 share classes (A, I, R, Z), all on the single snapshot date 2026-07-10, so every fund has exactly four classes.

**Links to accounts and positions.** We build the pair `fund_id + share_class` on both sides and count coverage both ways. Then we follow positions to a fund: each position row looks up its account (many to one, so the row count must not change). A position whose account is missing from `accounts` cannot reach any fund or ledger line.

In [ ]:
def fund_class(table):
    return table['fund_id'] + ' | ' + table['share_class']

display(coverage(fund_class(accounts), fund_class(ledger_nav), 'accounts (fund + class)', 'ledger_nav (fund + class)'))
display(coverage(accounts['fund_id'], ledger_nav['fund_id'], 'accounts fund_id', 'ledger_nav fund_id'))

account_funds = accounts[['account_id', 'fund_id', 'share_class']]
position_funds = positions[['account_id']].merge(account_funds, on='account_id', how='left', validate='many_to_one')
print(f'position rows before: {len(positions):,} | after looking up the account: {len(position_funds):,}')
positions_without_fund = int(position_funds['fund_id'].isna().sum())
print(f'position rows that reach no fund (account missing from accounts): {positions_without_fund:,}')

**What this tells us.** The fund and class pairs in `accounts` and `ledger_nav` match exactly in both directions (2,000 of 2,000), and so do the 500 fund IDs. The weak point is upstream: the lookup keeps all 480,663 position rows, but **699 of them reach no fund**, because their account is one of the 195 missing from `accounts`. Holdings that cannot be assigned to a fund cannot be part of any fund's NAV.

### 7.5 What looks suspicious in the four small tables

The list below is built from the numbers computed above, not typed in. These are observations to investigate, not confirmed defects; a few may turn out to be conventions of the synthetic data.

In [ ]:
usd_rows_not_one = int((usd['rate'] != 1).sum())
xyz_like = sorted(set(prices['currency']) - set(fx['currency']))   # price currencies with no FX rate at all
incomplete_grid = len(incomplete)

suspicious_small_tables = [
    f"{target_summary['events whose target is missing from securities']} corporate actions point to a target "
    f"security that is not in securities ({target_summary['...of which SPINOFF']} of them SPINOFF); "
    f"{target_summary['missing targets traded and held']} of these targets are traded and held "
    f"({target_summary['transaction rows on missing targets']:,} trades, "
    f"{target_summary['position rows on missing targets']:,} positions, "
    f"{target_summary['price rows on missing targets']} prices).",
    *[f"{row['action_type']}: {row['field']} {row['issue']} on {row['rows affected']} of {row['rows of type']} rows."
      for _, row in field_surprises.iterrows()],
    f"{ex_after_effective} of {len(corporate_actions)} corporate actions have ex_date after effective_date.",
    f"{incomplete_grid} securities have an incomplete price grid (fewer than {n_vendors} x {n_price_dates} rows); "
    f"the missing prices are: " + ', '.join(f"{vendor} on {day} ({count})" for (vendor, day), count
                                            in missing_vendor_dates.groupby(['vendor', 'missing date']).size().items()) + ".",
    f"{len(held_without_price)} held securities have no price at all.",
    f"Price currencies with no FX rate: {', '.join(xyz_like) or 'none'} "
    f"({int(prices['currency'].isin(xyz_like).sum()):,} price rows); {len(no_fx_securities)} securities carry such a "
    f"currency and are held in {int(positions['security_id'].isin(no_fx_securities).sum()):,} positions.",
    f"The USD rate is not exactly 1 on {usd_rows_not_one} of {len(usd)} dates "
    f"({usd['rate'].min():.6f} to {usd['rate'].max():.6f}).",
    f"{positions_without_fund:,} position rows reach no fund because their account is missing from accounts.",
]
for line in suspicious_small_tables:
    print('-', line)

**Summary of section 7.** The four small tables are well formed at their grain: every key is unique, every date is ISO, and the only blanks are expected ones. The problems are in their content and links:

- **Corporate actions are not usable as they are.** 23 spin-offs point to targets that do not exist; 15 of these are traded and held (1,131 trades, 1,118 positions) and never priced. All 31 mergers lack a target. Every row carries both a ratio and a cash rate, even where the event type has no use for them. 87 of 200 have the ex-date after the effective date.
- **Prices:** the preferred vendor has no price on the snapshot date for 30 securities. The 15 unpriced held securities are the spin-off targets. 12 held securities use the currency code XYZ, which has no FX rate.
- **FX:** the USD rate is never exactly 1.
- **Ledger:** the fund and class links are complete, but 699 position rows reach no fund because of orphan accounts.

<a id="statuses"></a>
## 8. Statuses and linked records in `transactions`

Each transaction has a `status` (BOOKED, CANCELLED or REVERSED), and some have a `related_transaction_id` that points to another transaction. Before a position rebuild we need to know what these links mean, because they decide which rows count.

The link can point in either direction (for example from a replacement back to the original it replaces), so we look at **every** status, not only CANCELLED and REVERSED. First: how many rows of each status, and how often is the link filled?

In [ ]:
has_link = transactions['related_transaction_id'] != ''
status_links = pd.DataFrame({
    'rows': transactions['status'].value_counts(),
    'related_transaction_id filled': has_link.groupby(transactions['status']).sum(),
})
status_links['share filled'] = (status_links['related_transaction_id filled'] / status_links['rows']).round(4)
status_links

**What this tells us.** 1,403,784 rows are BOOKED, 4,347 CANCELLED and 1,402 REVERSED. **Every REVERSED row has a link**, but only 842 of 4,347 CANCELLED rows (19%) do, and 3,511 BOOKED rows have one too. A link on a BOOKED row cannot mean "this row was reversed", so the field is used for more than one purpose.

**Following the links.** We take every row with a filled link and look up the record it points to (a left merge, so a link to an ID that does not exist stays visible). Section 5 showed that 2,105 IDs appear twice. If a link points to one of those, the merge returns **two** target rows for one link row. We print the rows before and after to see whether that happens, and how many links point to a repeated ID.

In [ ]:
target_columns = ['transaction_id', 'status', 'account_id', 'security_id', 'transaction_type', 'trade_date',
                  'settlement_date', 'quantity', 'price', 'currency', 'gross_amount', 'source_system',
                  'event_timestamp']
links = transactions.loc[has_link].copy()
links['row_number'] = links.index
targets = transactions[target_columns].add_prefix('target_')

links_expanded = links.merge(targets, left_on='related_transaction_id', right_on='target_transaction_id',
                             how='left', indicator=True)
repeated_ids = copies_in_txn[copies_in_txn > 1].index      # from section 4
print(f'link rows: {len(links):,} | rows after the merge: {len(links_expanded):,} '
      f'(+{len(links_expanded) - len(links):,})')
print(f'link rows that point to a repeated ID: {int(links["related_transaction_id"].isin(repeated_ids).sum())}')
print('link rows whose target exists / does not exist:')
print(links_expanded.drop_duplicates('row_number')['_merge']
      .map({'both': 'target exists', 'left_only': 'target missing'}).value_counts().to_string())

**What this tells us.** The merge adds 4 rows (5,755 to 5,759), because 4 link rows point to an ID that appears twice. The rows therefore have to be compared with one copy per target, otherwise those links would be counted twice. 4,913 links reach an existing record; **842 point to an ID that does not exist**.

To compare each link with **one** target, we keep the first copy (in file order) of a repeated target. That is only safe if the two copies agree in the columns we compare, so we check that first, after normalising the dates. Then we merge again with `validate='many_to_one'`, which makes pandas stop with an error if any target still appears twice, and we label each link by its kind: *status of the row -> status of the record it points to*.

In [ ]:
linked_repeats = links.loc[links['related_transaction_id'].isin(repeated_ids), 'related_transaction_id'].unique()
compared = [column for column in target_columns if column != 'transaction_id']
copies_agree = (normalise_transaction_dates(copy_a.loc[linked_repeats, compared])
                == normalise_transaction_dates(copy_b.loc[linked_repeats, compared])).all(axis=1)
print(f'repeated IDs that are a link target: {len(linked_repeats)} | '
      f'their two copies agree in all compared columns: {int(copies_agree.sum())}')

first_targets = targets.drop_duplicates('target_transaction_id', keep='first')
link_pairs = links.merge(first_targets, left_on='related_transaction_id', right_on='target_transaction_id',
                         how='left', validate='many_to_one')
print(f'link rows: {len(links):,} | rows after the merge with one copy per target: {len(link_pairs):,}')

link_pairs['link_kind'] = link_pairs['status'] + ' -> ' + link_pairs['target_status'].fillna('(ID not found)')
link_kinds = link_pairs['link_kind'].value_counts()
display(link_kinds.rename('link rows').to_frame())

# How many link rows point to the same target?
refs_per_target = link_pairs.groupby(['link_kind', 'related_transaction_id']).size()
print('Link rows per target (kind, rows pointing to one target -> number of targets):')
display(refs_per_target.groupby(level='link_kind').value_counts()
        .rename_axis(['link_kind', 'link rows pointing to one target']).rename('targets').reset_index())

# Targets with two link rows: are the two link rows one repeated ID (section 5) or two different records?
multi = refs_per_target[refs_per_target > 1].reset_index()[['link_kind', 'related_transaction_id']]
linkers = link_pairs.merge(multi, on=['link_kind', 'related_transaction_id'])
distinct_linkers = linkers.groupby(['link_kind', 'related_transaction_id'])['transaction_id'].nunique()
print('Targets with more than one link row, by number of distinct linking IDs (kind, IDs -> targets):')
distinct_linkers.groupby(level='link_kind').value_counts().to_dict()

**What this tells us.** The three repeated IDs that are link targets have identical copies in every compared column, so keeping the first copy loses nothing, and the second merge keeps exactly 5,755 rows. The data shows exactly three kinds of link, each pointing in one fixed direction:

- **REVERSED -> BOOKED** (1,402 rows): every original is referenced by **exactly two** REVERSED rows (701 originals).
- **BOOKED -> CANCELLED** (3,511 rows): a BOOKED row points back to the CANCELLED record it replaces, the "rebook" the Guide mentions. 3,489 cancellations have one rebook. For 11 there are two link rows: in 4 cases they are the two copies of one repeated ID (section 5), in 7 cases two different BOOKED records.
- **CANCELLED -> (ID not found)** (842 rows): see below.

No REVERSED row points to a CANCELLED one, and no link points to a REVERSED row.

**How does a linked row compare with its target?** For each kind of link (where the target exists) we count:

- whether account, security, transaction_type, settlement_date and source_system are the same;
- how `quantity` and `gross_amount` compare: **equal**, **opposite sign** (one is minus the other), **different**, or **blank** on either side;
- the gap in trade dates in days (both normalised first) and the gap in `event_timestamp` in minutes (linked row minus target).

The comparison labels use the same simple rules as section 5.

In [ ]:
def number_relation(left_text, right_text):
    """Label each pair of numbers written as text: blank, equal, opposite sign or different."""
    left = pd.to_numeric(left_text, errors='coerce')
    right = pd.to_numeric(right_text, errors='coerce')
    label = pd.Series('different', index=left_text.index)
    label[left == -right] = 'opposite sign'
    label[left == right] = 'equal'                      # checked after, so 0 and -0 count as equal
    label[left.isna() | right.isna()] = 'blank or not a number'
    return label


found = link_pairs[link_pairs['target_status'].notna()].copy()
found['quantity_relation'] = number_relation(found['quantity'], found['target_quantity'])
found['price_relation'] = number_relation(found['price'], found['target_price'])
found['gross_relation'] = number_relation(found['gross_amount'], found['target_gross_amount'])
found['trade_date_gap_days'] = (
    pd.to_datetime(normalise_date(found['trade_date']), format='%Y-%m-%d', errors='coerce')
    - pd.to_datetime(normalise_date(found['target_trade_date']), format='%Y-%m-%d', errors='coerce')).dt.days
found['event_gap_minutes'] = (
    pd.to_datetime(found['event_timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    - pd.to_datetime(found['target_event_timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
).dt.total_seconds() / 60

link_profile = {}
for kind, group in found.groupby('link_kind'):
    profile = {'link rows': len(group)}
    for column in ['account_id', 'security_id', 'transaction_type', 'settlement_date', 'source_system']:
        profile[f'same {column}'] = int((group[column] == group[f'target_{column}']).sum())
    for relation_column, name in [('quantity_relation', 'quantity'), ('price_relation', 'price'),
                                  ('gross_relation', 'gross_amount')]:
        for label in ['equal', 'opposite sign', 'different', 'blank or not a number']:
            profile[f'{name}: {label}'] = int((group[relation_column] == label).sum())
    for gap_column, name in [('trade_date_gap_days', 'trade date gap (days)'),
                             ('event_gap_minutes', 'event_timestamp gap (minutes)')]:
        profile[f'{name}: min / median / max'] = (f"{group[gap_column].min():g} / {group[gap_column].median():g} / "
                                                   f"{group[gap_column].max():g}")
        profile[f'{name}: not readable'] = int(group[gap_column].isna().sum())
    link_profile[kind] = profile
link_profile = pd.DataFrame(link_profile)
link_profile

**What this tells us.** The links always stay on the same `transaction_type` and `source_system`, and the trade date and event time are **exactly** the same (gap 0 in every row). The two kinds then behave differently:

- **REVERSED -> BOOKED:** the REVERSED row repeats the original. The account and security are the same in all 1,402 rows, and the quantity is equal in 1,392. It is opposite in sign in 3 cases, where one side has a negative quantity, and blank in 7. **No REVERSED row has the opposite sign by design.**
- **BOOKED -> CANCELLED:** the rebook keeps the account and security in nearly all rows (3,504 and 3,505 of 3,511), the price in 3,501 and the gross amount in all 3,511. But **the quantity differs in 3,336 of 3,511**.

A rebook that changes the quantity but not the gross amount is odd, so the next cell looks at it.

**Which quantity does `gross_amount` belong to?** For each link where the quantity differs from the target's, we ask which side's `quantity x price` reproduces the linked row's `gross_amount`, within the tolerance in `config/exploration.toml` (gross is stored to the cent). We also show how large the quantity change is, as the ratio of the linked quantity to the target quantity.

In [ ]:
GROSS_TOLERANCE = CONFIG['reconstruction']['gross_amount_tolerance']
changed = found[found['quantity_relation'] == 'different'].copy()
gross = pd.to_numeric(changed['gross_amount'], errors='coerce')
own_fit = (gross - pd.to_numeric(changed['quantity'], errors='coerce')
           * pd.to_numeric(changed['price'], errors='coerce')).abs() <= GROSS_TOLERANCE
target_fit = (gross - pd.to_numeric(changed['target_quantity'], errors='coerce')
              * pd.to_numeric(changed['target_price'], errors='coerce')).abs() <= GROSS_TOLERANCE
changed['quantity_ratio'] = (pd.to_numeric(changed['quantity'], errors='coerce')
                             / pd.to_numeric(changed['target_quantity'], errors='coerce'))
gross_fit = changed.assign(own_fit=own_fit, target_fit=target_fit).groupby('link_kind').agg(
    rows_with_changed_quantity=('quantity', 'size'),
    gross_fits_own_quantity=('own_fit', 'sum'),
    gross_fits_target_quantity=('target_fit', 'sum'),
    ratio_min=('quantity_ratio', 'min'), ratio_median=('quantity_ratio', 'median'),
    ratio_max=('quantity_ratio', 'max'))
gross_fit

**What this tells us.** In 3,328 of the 3,336 rebooks with a changed quantity, the rebook's `gross_amount` is the *cancelled original's* quantity times price, never its own (0 rows). The new quantity is typically within about 5 to 10% of the old one (ratio median 0.99, from 0.95 to 1.1, plus one negative ratio from a sign flip). This suggests the rebook corrected the quantity and **copied the gross amount from the original without recalculating it**. Which quantity is right cannot be decided from this table alone.

**The REVERSED rows that share a target.** The previous table showed how many REVERSED rows point to each original. Where two REVERSED rows point to the same original, we compare the two rows with each other (the first and second in file order), after normalising their dates: which columns differ, apart from their own `transaction_id`? We also check how far apart they sit in the file.

In [ ]:
reversed_rows = transactions[transactions['status'] == 'REVERSED'].copy()
reversed_rows['row_number'] = reversed_rows.index
reversed_rows['copy_number'] = reversed_rows.groupby('related_transaction_id').cumcount()
reversals_per_target = reversed_rows.groupby('related_transaction_id').size()
print('REVERSED rows per original (rows -> originals):', reversals_per_target.value_counts().to_dict())

two_reversals = reversals_per_target[reversals_per_target == 2].index
first_reversal = normalise_transaction_dates(
    reversed_rows[reversed_rows['copy_number'] == 0].set_index('related_transaction_id').loc[two_reversals])
second_reversal = normalise_transaction_dates(
    reversed_rows[reversed_rows['copy_number'] == 1].set_index('related_transaction_id').loc[two_reversals])
reversal_columns = [column for column in data_columns if column not in ('transaction_id', 'related_transaction_id')]
reversal_differs = pd.DataFrame({column: first_reversal[column] != second_reversal[column]
                                 for column in reversal_columns})
print(f'originals with two REVERSED rows: {len(two_reversals)} | '
      f'pairs identical in every column except the ID: {int((~reversal_differs.any(axis=1)).sum())}')
print('Pairs that differ, per column:')
print(reversal_differs.sum()[reversal_differs.sum() > 0].to_string())
reversal_row_gap = second_reversal['row_number'] - first_reversal['row_number']
print('Rows between the two REVERSED rows: min', reversal_row_gap.min(), '| median', reversal_row_gap.median(),
      '| max', reversal_row_gap.max())

**What this tells us.** All 701 originals have exactly two REVERSED rows. After date normalisation, 688 of the pairs are identical in every column except their own ID; 13 differ in settlement_date (6), quantity (4) or price (3). The two rows are not adjacent in the file (median 34 rows apart, up to 134). So these are two separate records, not one record sent twice like the section 5 duplicates. **Applying each of them as an offset would reverse the original twice.**

**The links that lead nowhere.** Some CANCELLED rows point to an ID that is not in `transactions`. We count them and show three, picked by a rule fixed before looking: the first three in sorted `transaction_id` order.

In [ ]:
dangling = link_pairs[link_pairs['target_status'].isna()]
print(f'link rows whose target ID is not in transactions: {len(dangling):,}')
print('their status:', dangling['status'].value_counts().to_dict())
print(f'distinct missing target IDs: {dangling["related_transaction_id"].nunique():,}')
cancelled_rows = transactions[transactions['status'] == 'CANCELLED']
cancelled_unlinked = cancelled_rows[cancelled_rows['related_transaction_id'] == '']
rebooked_ids = link_pairs.loc[link_pairs['link_kind'] == 'BOOKED -> CANCELLED', 'related_transaction_id']
print(f'CANCELLED rows without a link of their own: {len(cancelled_unlinked):,} | '
      f'...that a BOOKED rebook points to: {int(cancelled_unlinked["transaction_id"].isin(rebooked_ids).sum()):,}')
print(f'CANCELLED rows with a dangling link that a rebook points to: '
      f'{int(cancelled_rows.loc[cancelled_rows["related_transaction_id"] != "", "transaction_id"].isin(rebooked_ids).sum())}')
dangling.sort_values('transaction_id').head(3)[['transaction_id', 'status', 'related_transaction_id',
                                                 'account_id', 'security_id', 'transaction_type',
                                                 'trade_date', 'quantity', 'event_timestamp']]

**What this tells us.** All 842 links that lead nowhere start from a CANCELLED row, and each points to a different missing ID. These are cancellations whose original was never loaded, or whose ID was changed. The brief lists "cancels with no matching original" among the expected defects. The other 3,505 CANCELLED rows have no link of their own, but a BOOKED rebook points to 3,502 of these rows, so for them the link sits on the replacement. 3 CANCELLED rows have neither their own link nor a rebook, and no cancellation with a dangling link has been rebooked.

### 8.1 What does REVERSED mean?

The data dictionary describes `status` as: *"BOOKED, CANCELLED or REVERSED; investigate linked records and **distinguish state from offsetting-event interpretations**."* The Dataset Guide (section 3) says: *"REVERSED may describe an inactive transaction state or an offsetting event; the extract does not supply an explicit event-operation field. Compare plausible interpretations, prevent double offsets, and identify what additional contract would resolve the ambiguity."* It also says that *"a BOOKED replacement linked to a CANCELLED original can represent a valid rebook."*

Four readings are possible for a REVERSED row and the BOOKED original it points to. For a position rebuild they mean:

| Reading | What counts in the position |
|---|---|
| **R1, state (the reversal record is inactive)** | the original counts, REVERSED rows do not (this is BOOKED-only) |
| **R2, state (the original was reversed out)** | neither the original nor the REVERSED rows count |
| **R3, offsetting event, once per original** | the original counts and **one** REVERSED row subtracts it, so the pair nets to zero |
| **R4, offsetting event, every row** | the original counts and **each** REVERSED row subtracts it |

The next cell collects the evidence from this section in one table. Section 9 then shows each reading's effect on the rebuild. We do not choose a treatment here.

In [ ]:
reversed_kind = 'REVERSED -> BOOKED'
rev = link_profile[reversed_kind]
offsets_per_original = reversals_per_target.value_counts().to_dict()
evidence = pd.DataFrame([
    {'evidence': 'REVERSED rows with a link to an existing BOOKED original',
     'value': f"{link_kinds.get(reversed_kind, 0):,} of {int(status_links.loc['REVERSED', 'rows']):,}"},
    {'evidence': 'REVERSED rows per original (rows -> originals)', 'value': str(offsets_per_original)},
    {'evidence': 'same account / security / transaction_type as the original',
     'value': f"{rev['same account_id']:,} / {rev['same security_id']:,} / {rev['same transaction_type']:,}"},
    {'evidence': 'quantity: equal / opposite sign / different / blank',
     'value': (f"{rev['quantity: equal']:,} / {rev['quantity: opposite sign']:,} / {rev['quantity: different']:,} / "
               f"{rev['quantity: blank or not a number']:,}")},
    {'evidence': 'trade date gap (days), event_timestamp gap (minutes): min / median / max',
     'value': f"{rev['trade date gap (days): min / median / max']}; {rev['event_timestamp gap (minutes): min / median / max']}"},
    {'evidence': 'a field that says whether a row is a state or an operation', 'value': 'none in the extract'},
])
evidence

**What this tells us.** The evidence does **not** settle the meaning of REVERSED. Each reading has support and a problem:

- **R1 (the reversal record is inactive, the original stays):** fits the fact that REVERSED rows carry the original's own sign, type and quantity, like a state record rather than a counter-trade. Problem: the original is then never reversed in the position, and its own status still says BOOKED.
- **R2 (the original was reversed out):** treats the REVERSED rows as markers that the original no longer counts. Problem: nothing on the original says so; we infer it from rows that point to it.
- **R3 (one offset per original):** the usual meaning of a reversal in accounting, and it avoids a double offset. Problems: the rows have the *same* sign as the original, so we would have to impose the opposite sign. There are two rows per original, so picking one is arbitrary (13 of 701 pairs differ). In quantity, R3 equals R2 except where the two REVERSED rows disagree.
- **R4 (every row offsets):** the original ends at minus its own quantity. This is exactly the double offset the Guide warns against. We keep it only as evidence, because section 9 tests whether the book of record follows it.

**What would resolve it:** an explicit event-operation field (state change versus offsetting entry), a stated sign convention for offsets, and an explanation of why each original has two REVERSED records. This is our faculty question on REVERSED. Section 9 shows every reading side by side.

<a id="rebuild"></a>
## 9. Dry-run reconstruction: rebuilding positions from trades

Can we rebuild `positions` from `transactions`? This is a **dry run** to learn how the data behaves, not the final reconciliation. It is **trade-only**: no corporate action is applied, as the Dataset Guide asks (event-adjusted scenarios stay separate so that no quantity moves twice).

**Definitions (fixed before any code runs):**

- **Opening balance:** zero before 1 June, the Guide's shared benchmark (a convention, not a claim that portfolios started empty).
- **Signed quantity:** `+quantity` for BUY and `-quantity` for SELL, using the raw `quantity` exactly as it is (a negative raw quantity stays negative).
- **Key:** `account_id + security_id`. To compare with `positions` and `comparison_positions` we add `as_of_date` = the snapshot date from `config/exploration.toml`, because both reported tables use the full key `account_id + security_id + as_of_date`.
- **Match:** the rebuilt and reported quantities, both converted to numbers, differ by no more than the tolerance in `config/exploration.toml` (one millionth of a unit). The tolerance only absorbs float noise: 869 trade quantities are stored like `3551.2200000000003`. Every table also shows how many matches are **exact**.

**Groups.** Every key of the full outer comparison lands in exactly one group, and the code checks that the groups add up to the number of keys:

| Group | Meaning |
|---|---|
| matched | in both tables, evaluable, difference within tolerance |
| different | in both tables, evaluable, difference above tolerance |
| only in rebuild, nonzero | trades exist, the reported table has no row, and the trades do not net to zero |
| only in rebuild, nets to zero | trades exist and net to zero, and the reported table has no row. **Kept as its own group**: it is neither a match nor a break until we decide |
| only in reported table | the reported table has a row, but no trade contributes to it |
| not evaluable | a contributing trade has no usable quantity (blank or unreadable), or the variant marks the key (for example a conflicting duplicate). Split into "in both" and "only in rebuild" |

A key that is only in the reported table is always counted there, whatever its quantity, because there is nothing to evaluate.

**Match rates** always name their denominator. **Rate A** = matched / (matched + different), out of the keys that could be compared. **Rate B** = matched / all keys in the union.

**Preparing the trades.** We take the columns the rebuild needs and convert types explicitly, counting every value that fails:

- `trade_date` and `settlement_date` are normalised with `normalise_date` (9,398 trade dates are DD/MM/YYYY), then converted to real dates;
- `quantity` is converted to a number, keeping blanks and unreadable values as missing (never zero);
- the signed quantity follows the definition above.

We also attach, from earlier sections, the facts each step needs: the pair type and copy number of repeated IDs (section 5), whether a BOOKED row is the original of a reversal, and whether a REVERSED row is the first one for its original (section 8).

In [ ]:
trades = transactions[['transaction_id', 'account_id', 'security_id', 'transaction_type', 'status',
                       'related_transaction_id', 'quantity', 'price', 'gross_amount']].copy()
trades['row_number'] = trades.index                                   # lineage only, never a key
trades['trade_day'] = pd.to_datetime(normalise_date(transactions['trade_date']), format='%Y-%m-%d', errors='coerce')
trades['settlement_day'] = pd.to_datetime(normalise_date(transactions['settlement_date']), format='%Y-%m-%d',
                                          errors='coerce')
trades['quantity_number'] = pd.to_numeric(trades['quantity'], errors='coerce')
trades['quantity_blank'] = trades['quantity'] == ''
direction = trades['transaction_type'].map({'BUY': 1, 'SELL': -1})      # missing for any other type
trades['signed_quantity'] = direction * trades['quantity_number']

pair_type = pd.Series('exact copy', index=copy_a.index)                 # one label per repeated ID
pair_type[pair_differs_only_by_format] = 'differs only by date format'
pair_type[pair_has_real_conflict] = 'real conflict'
trades['pair_type'] = trades['transaction_id'].map(pair_type).fillna('not repeated')
trades['copy_number'] = trades.groupby('transaction_id').cumcount()   # 0 = first copy in file order

is_reversed = trades['status'] == 'REVERSED'
trades['is_reversed_original'] = ((trades['status'] == 'BOOKED')
                                  & trades['transaction_id'].isin(trades.loc[is_reversed, 'related_transaction_id']))
trades['is_first_reversal'] = is_reversed & ~trades['related_transaction_id'].where(is_reversed).duplicated()

conversion_report = pd.Series({
    'rows': len(trades),
    'trade_date blank': int((transactions['trade_date'] == '').sum()),
    'trade_date not readable (not blank)': int((trades['trade_day'].isna() & (transactions['trade_date'] != '')).sum()),
    'settlement_date blank': int((transactions['settlement_date'] == '').sum()),
    'settlement_date not readable (not blank)': int((trades['settlement_day'].isna()
                                                     & (transactions['settlement_date'] != '')).sum()),
    'quantity blank': int(trades['quantity_blank'].sum()),
    'quantity not readable (not blank)': int((trades['quantity_number'].isna() & ~trades['quantity_blank']).sum()),
    'quantity negative': int((trades['quantity_number'] < 0).sum()),
    'transaction_type not BUY or SELL': int(direction.isna().sum()),
    'BOOKED rows that are a reversed original': int(trades['is_reversed_original'].sum()),
    'REVERSED rows that are the first for their original': int(trades['is_first_reversal'].sum()),
})
display(conversion_report.to_frame('count'))
trades['pair_type'].value_counts().to_frame('rows')

**What this tells us.** All 1,409,533 rows convert cleanly: every trade date is readable, and the only gaps are the ones we already knew. There are 1,687 blank settlement dates, 2,819 blank quantities and 655 negative quantities, and every type is BUY or SELL. The pair labels agree with section 5: 4,064 rows in exact pairs, 56 in format-only pairs, 90 in conflicting pairs. 702 BOOKED rows are the original of a reversal (701 IDs, one of which is a repeated ID with two copies), and 701 REVERSED rows are the first for their original.

**The helpers.** Four small functions do all the work, so every variant below is computed the same way:

- `rebuild(rows)` sums `quantity_used` per key and counts the contributing rows. A key is not evaluable if any of its rows has no number, or if a row is marked by the variant.
- `compare_groups(rebuilt, reported)` makes the full outer merge on `account_id + security_id + as_of_date` (`validate='one_to_one'`, so a repeated key would stop the code) and puts every key in one group.
- `summarise(merged)` counts the groups, checks that they add up to the number of keys, and computes both match rates.
- `run_variant(...)` does all of this against both reported tables and stores one line of results. It also records the `transaction_count` check, the 480 book-only positions and the orphan keys, which we look at after the steps.

The reported quantities are converted to numbers once; failures are counted.

In [ ]:
KEY = ['account_id', 'security_id']
CUTOFF = pd.Timestamp(AS_OF_DATE)
GROUPS = ['matched', 'different', 'only in rebuild, nonzero', 'only in rebuild, nets to zero',
          'only in reported table', 'not evaluable (in both)', 'not evaluable (only in rebuild)']
RATE_A = 'rate A: matched / (matched + different)'
RATE_B = 'rate B: matched / keys in union'

reported_tables = {}
for reported_name, table in [('positions', positions), ('comparison_positions', comparison_positions)]:
    side = table[position_key].copy()
    side['reported_quantity'] = pd.to_numeric(table['quantity'], errors='coerce')
    if 'transaction_count' in table.columns:
        side['transaction_count'] = pd.to_numeric(table['transaction_count'], errors='coerce')
    reported_tables[reported_name] = side
    print(f"{reported_name}: {len(side):,} rows | quantity not readable: {int(side['reported_quantity'].isna().sum())}"
          f" | as_of_date values: {sorted(side['as_of_date'].unique())}")
print('positions transaction_count not readable:', int(reported_tables['positions']['transaction_count'].isna().sum()))

book_only_keys = pd.MultiIndex.from_frame(book_only[position_key])   # the 480 keys from section 6


def rebuild(rows):
    """One row per key: the summed quantity, the contributing rows, and whether the key is evaluable."""
    rows = rows.assign(no_number=rows['quantity_used'].isna())
    rebuilt = rows.groupby(KEY, as_index=False).agg(
        rebuilt_quantity=('quantity_used', 'sum'),       # sum skips missing values; those keys are flagged below
        contributing_rows=('quantity_used', 'size'),
        rows_without_number=('no_number', 'sum'),
        marked=('mark_not_evaluable', 'any'))
    rebuilt['not_evaluable'] = (rebuilt['rows_without_number'] > 0) | rebuilt['marked']
    return rebuilt


def compare_groups(rebuilt, reported):
    """Full outer comparison of a rebuild with one reported table; every key gets one group."""
    merged = rebuilt.assign(as_of_date=AS_OF_DATE).merge(reported, on=position_key, how='outer',
                                                         indicator=True, validate='one_to_one')
    in_both = merged['_merge'] == 'both'
    only_rebuild = merged['_merge'] == 'left_only'
    not_evaluable = merged['not_evaluable'].eq(True) | (in_both & merged['reported_quantity'].isna())
    merged['difference'] = merged['rebuilt_quantity'] - merged['reported_quantity']
    within = merged['difference'].abs() <= TOLERANCE
    nets_to_zero = merged['rebuilt_quantity'].abs() <= TOLERANCE

    group = pd.Series('', index=merged.index)
    group[merged['_merge'] == 'right_only'] = 'only in reported table'
    group[only_rebuild & not_evaluable] = 'not evaluable (only in rebuild)'
    group[only_rebuild & ~not_evaluable & nets_to_zero] = 'only in rebuild, nets to zero'
    group[only_rebuild & ~not_evaluable & ~nets_to_zero] = 'only in rebuild, nonzero'
    group[in_both & not_evaluable] = 'not evaluable (in both)'
    group[in_both & ~not_evaluable & within] = 'matched'
    group[in_both & ~not_evaluable & ~within] = 'different'
    assert (group != '').all(), 'a key was left without a group'
    merged['group'] = group
    merged['exact_match'] = (group == 'matched') & (merged['difference'] == 0)
    return merged


def summarise(merged):
    """Group counts, exact matches and both match rates for one comparison."""
    counts = merged['group'].value_counts().reindex(GROUPS, fill_value=0)
    assert counts.sum() == len(merged), 'groups do not add up to the keys in the union'
    summary = {group: int(counts[group]) for group in GROUPS}
    summary['matched exactly'] = int(merged['exact_match'].sum())
    summary['keys in union'] = len(merged)
    comparable = counts['matched'] + counts['different']
    summary[RATE_A] = round(counts['matched'] / comparable, 6) if comparable else float('nan')
    summary[RATE_B] = round(counts['matched'] / len(merged), 6)
    return summary


step_results = {}     # variant name -> one line of results, filled by run_variant
kept_comparisons = {} # variant name -> full comparison with positions, only for the variants we trace later


def run_variant(step, name, kind, rows, keep=False):
    """Rebuild from `rows`, compare with both reported tables and store the results under `name`."""
    rebuilt = rebuild(rows)
    record = {'step': step, 'variant': name, 'kind': kind, 'rows used': len(rows), 'keys rebuilt': len(rebuilt),
              'keys with a negative quantity': int(rows.loc[rows['quantity_number'] < 0, KEY].drop_duplicates().shape[0]),
              'keys with a blank quantity': int(rows.loc[rows['quantity_blank'], KEY].drop_duplicates().shape[0])}
    for reported_name, reported in reported_tables.items():
        merged = compare_groups(rebuilt, reported)
        record[reported_name] = summarise(merged)
        if reported_name == 'positions':
            in_both = merged['_merge'] == 'both'
            record['keys in both'] = int(in_both.sum())
            record['transaction_count equal'] = int((in_both & (merged['contributing_rows']
                                                                == merged['transaction_count'])).sum())
            is_book_only = merged.set_index(position_key).index.isin(book_only_keys)
            record['book-only 480'] = merged.loc[is_book_only, 'group'].value_counts().to_dict()
            is_orphan = (merged['account_id'].isin(orphan_account_ids)
                         | merged['security_id'].isin(orphan_security_ids))
            record['orphan keys'] = merged.loc[is_orphan, 'group'].value_counts().to_dict()
            if keep:
                kept_comparisons[name] = merged
    step_results[name] = record
    return record


def step_table(names, reported_name):
    """One row per variant: rows used, the groups and the two rates against one reported table."""
    lines = []
    for name in names:
        record = step_results[name]
        lines.append({'step': record['step'], 'variant': name, 'kind': record['kind'],
                      'rows used': record['rows used'], **record[reported_name]})
    return pd.DataFrame(lines).set_index('variant')


def show_step(names):
    """Display one step's variants against both reported tables (one column per variant)."""
    for reported_name in reported_tables:
        print(f'against {reported_name}:')
        display(step_table(names, reported_name).T)

**What this tells us.** Both reported tables have readable quantities and a single snapshot date, 2026-07-10, the same as the date in config. So adding `as_of_date` to the rebuild key is safe, and every key of the reported tables can be compared. `transaction_count` is readable on every row.

**The steps as small functions.** Each transformation is one function that takes trade rows and returns trade rows. A variant is a list of steps applied in order, so "applied alone" and "applied after earlier steps" use exactly the same code. `quantity_used` starts as the raw signed quantity, and `mark_not_evaluable` starts as False. The order of the chain is fixed here, before any result is seen: **a** duplicates, **b** conflicting duplicates, **c** statuses, **d** negative quantities, **e** blank quantities, **f** the cut-off date.

In [ ]:
def raw_rows(rows):
    """C0: every row, raw signed quantity, nothing marked."""
    return rows.assign(quantity_used=rows['signed_quantity'], mark_not_evaluable=False)

# a. duplicates (section 5)
def drop_second_copies(rows, pair_types):
    """Drop every copy after the first of a repeated ID whose pair type is listed."""
    return rows[~((rows['copy_number'] >= 1) & rows['pair_type'].isin(pair_types))]

def drop_exact_and_format_copies(rows):
    return drop_second_copies(rows, ['exact copy', 'differs only by date format'])

def drop_exact_copies_only(rows):
    return drop_second_copies(rows, ['exact copy'])

# b. conflicting duplicates: four options
def mark_conflicts_not_evaluable(rows):
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | (rows['pair_type'] == 'real conflict'))

def keep_first_conflict_copy(rows):
    return drop_second_copies(rows, ['real conflict'])

def keep_second_conflict_copy(rows):
    return rows[~((rows['pair_type'] == 'real conflict') & (rows['copy_number'] == 0))]

def drop_both_conflict_copies(rows):
    return rows[rows['pair_type'] != 'real conflict']

# c. statuses, and the four readings of REVERSED (section 8.1)
def drop_cancelled(rows):
    return rows[rows['status'] != 'CANCELLED']

def reversed_R1_drop_reversed(rows):
    """R1: the REVERSED rows do not count; the original stays."""
    return rows[rows['status'] != 'REVERSED']

def reversed_R2_drop_original_too(rows):
    """R2: neither the REVERSED rows nor the BOOKED original they point to count."""
    return rows[(rows['status'] != 'REVERSED') & ~rows['is_reversed_original']]

def reversed_R3_one_offset(rows):
    """R3: the first REVERSED row per original subtracts its quantity; the second one does not count."""
    rows = rows[(rows['status'] != 'REVERSED') | rows['is_first_reversal']]
    offset = rows['status'] == 'REVERSED'
    return rows.assign(quantity_used=rows['quantity_used'].where(~offset, -rows['quantity_used']))

def reversed_R4_every_offset(rows):
    """R4: every REVERSED row subtracts its quantity (a double offset when there are two per original)."""
    offset = rows['status'] == 'REVERSED'
    return rows.assign(quantity_used=rows['quantity_used'].where(~offset, -rows['quantity_used']))

# d. negative raw quantities
def negatives_kept_raw(rows):
    return rows                                    # nothing changes; the keys are counted in the step table

def negatives_not_evaluable(rows):
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | (rows['quantity_number'] < 0))

def negatives_as_absolute(rows):
    """Sensitivity only: treat a negative raw quantity as its absolute value. Never carried forward."""
    negative = rows['quantity_number'] < 0
    return rows.assign(quantity_used=rows['quantity_used'].where(~negative, -rows['quantity_used']))

# e. blank quantities
def blanks_not_evaluable(rows):
    return rows                                    # already the definition: a row without a number marks its key

def blanks_as_zero(rows):
    """Sensitivity only: a blank quantity counts as 0 units. Never carried forward."""
    return rows.assign(quantity_used=rows['quantity_used'].where(~rows['quantity_blank'], 0.0))

# f. the cut-off date
def trade_date_cutoff(rows):
    """Keep trades with trade_date on or before the snapshot; an unreadable date marks its key."""
    unknown = rows['trade_day'].isna()
    rows = rows[unknown | (rows['trade_day'] <= CUTOFF)]
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | rows['trade_day'].isna())

def settlement_date_cutoff(rows):
    """Keep trades with settlement_date on or before the snapshot; a blank or unreadable date marks its key."""
    unknown = rows['settlement_day'].isna()
    rows = rows[unknown | (rows['settlement_day'] <= CUTOFF)]
    return rows.assign(mark_not_evaluable=rows['mark_not_evaluable'] | rows['settlement_day'].isna())


def apply_steps(rows, steps):
    for step_function in steps:
        rows = step_function(rows)
    return rows


# The chain (the no-repair path), fixed in this order before looking at results.
CHAIN = [
    ('C0', 'C0 raw benchmark', [raw_rows]),
    ('a', 'C1 drop exact and format-only copies', [drop_exact_and_format_copies]),
    ('b', 'C2 conflicting keys not evaluable', [mark_conflicts_not_evaluable]),
    ('c1', 'C3 drop CANCELLED', [drop_cancelled]),
    ('c2', 'C4 drop REVERSED (R1, BOOKED-only)', [reversed_R1_drop_reversed]),
    ('d', 'C5 negatives kept raw', [negatives_kept_raw]),
    ('e', 'C6 blanks not evaluable', [blanks_not_evaluable]),
    ('f', 'C7 trade-date cut-off', [trade_date_cutoff]),
]
chain_steps = {}           # variant name -> the full list of steps from the raw trades
steps_so_far = []
for step, name, steps in CHAIN:
    steps_so_far = steps_so_far + steps
    chain_steps[name] = steps_so_far
chain_steps['C7s settlement-date cut-off'] = chain_steps['C6 blanks not evaluable'] + [settlement_date_cutoff]
print('chain:', ' -> '.join(chain_steps))

**What this tells us.** The chain is fixed: C0 raw, then a, b, c1, c2, d, e, f. The settlement-date version C7s branches off after C6. The same step functions also build the branches and the "alone" variants, so a difference between two variants can only come from the steps they use.

### 9.1 The raw benchmark (C0)

Every transaction row counts, whatever its status, and duplicates count twice. This is the starting point the Guide asks for: we will see the effect of each change against it. The three small run functions below say how a variant is built: **chain** (all chain steps up to here), **branch** (the chain up to the previous step, then another option), or **alone** (one step applied directly to C0).

In [ ]:
chain_step_of = {name: step for step, name, _ in CHAIN}
chain_step_of['C7s settlement-date cut-off'] = 'f'


def run_chain(name, keep=False):
    return run_variant(chain_step_of[name], name, 'chain', apply_steps(trades, chain_steps[name]), keep)


def run_branch(step, name, after, extra_steps):
    return run_variant(step, name, f'branch after {after.split()[0]}',
                       apply_steps(trades, chain_steps[after] + extra_steps))


def run_alone(step, name, extra_steps, keep=False):
    return run_variant(step, name, 'alone on C0', apply_steps(trades, [raw_rows] + extra_steps), keep)


run_chain('C0 raw benchmark')
show_step(['C0 raw benchmark'])

**What this tells us.** With every row counted, the full outer comparison has 480,870 keys.

- **Against `positions`:** 471,746 keys match (all but one exactly), 5,880 differ and 384 are only in positions. 8 keys are only in the rebuild with a nonzero quantity, and 42 more net to zero (their own group). 2,810 are not evaluable (2,653 in both, 157 only in the rebuild) because a contributing quantity is blank. Rate A is 98.77% of the 477,626 comparable keys; rate B is 98.10% of all 480,870 keys.
- **Against `comparison_positions`:** the picture is similar, except for 486 keys only in the rebuild with a nonzero quantity. This group contains the 480 holdings the custodian does not report. Rate A is 98.38%, rate B 97.61%.

### 9.2 Step a: exact duplicates

We drop every copy after the first of a repeated ID whose copies are identical. Section 5 found 2,032 exact pairs and 28 pairs that differ only in how `trade_date` is written. The chain (C1) treats both as the same record. The alternative counts only the 2,032 as exact and keeps both copies of the 28. Because this is the first step, "after C0" and "alone on C0" are the same thing here.

In [ ]:
run_chain('C1 drop exact and format-only copies')
run_alone('a', 'a alone: exact copies only (format-only pairs kept twice)', [drop_exact_copies_only])
show_step(['C0 raw benchmark', 'C1 drop exact and format-only copies',
           'a alone: exact copies only (format-only pairs kept twice)'])

**What this tells us.** Removing the duplicate copies takes out 2,060 rows, and the match **falls** by 2,009 keys against positions (1,998 against the custodian). Counting the 28 format-only pairs twice gives back 26 of those matches. So **both reported tables count the second copy of a duplicate**, as if it were a separate trade. This is a finding about the reported tables, not a reason to keep the duplicates: the Guide asks for the raw benchmark and the justified change side by side, and here the justified change moves us away from the book.

### 9.3 Step b: conflicting duplicates

For the 45 pairs whose copies disagree on a fact (section 5) we do not know which copy is right. There are four options: keep the first copy, keep the second, drop both, or keep both rows but mark the key **not evaluable**. The chain (C2) uses the last one, because it repairs nothing. The other three are shown after C1 and alone on C0. We do not pick one.

In [ ]:
run_chain('C2 conflicting keys not evaluable')
conflict_options = [('keep first copy', [keep_first_conflict_copy]),
                    ('keep second copy', [keep_second_conflict_copy]),
                    ('drop both copies', [drop_both_conflict_copies])]
for label, steps in conflict_options:
    run_branch('b', f'b after C1: {label}', 'C1 drop exact and format-only copies', steps)
run_alone('b', 'b alone: conflicting keys not evaluable', [mark_conflicts_not_evaluable])
for label, steps in conflict_options:
    run_alone('b', f'b alone: {label}', steps)

step_b_names = (['C1 drop exact and format-only copies', 'C2 conflicting keys not evaluable']
                + [f'b after C1: {label}' for label, _ in conflict_options]
                + ['b alone: conflicting keys not evaluable'] + [f'b alone: {label}' for label, _ in conflict_options])
show_step(step_b_names)

**What this tells us.** Marking the conflicting pairs not evaluable moves 39 keys into that group (37 had matched, 2 had differed). The other conflicting pairs were already not evaluable because one copy has a blank quantity. The three repair options are almost the same as each other: keep first 469,703 matches, keep second 469,702, drop both 469,701. All three are below C1 (469,737), because the book counts both copies here too. The choice touches about 45 keys out of 480,870, so it is a question of record correctness, not of the headline match rate. The same pattern appears when the options are applied alone on C0.

### 9.4 Step c: statuses (CANCELLED, then REVERSED)

First the CANCELLED rows leave (C3). A BOOKED rebook that points to a CANCELLED original stays, because it is the replacement. Then the REVERSED rows, under each reading from section 8.1:

- **R1** drops the REVERSED rows and keeps the original (the chain, C4). After C3 this is BOOKED-only.
- **R2** drops the REVERSED rows and the original.
- **R3** keeps the original and lets the first REVERSED row per original subtract it once.
- **R4** lets every REVERSED row subtract. This is a double offset; it is evidence only, not a candidate treatment.

The Guide's shared benchmark, **BOOKED rows only with nothing else changed**, is the column "c alone: BOOKED only (Guide shared benchmark)". It still contains every duplicate. We also apply the whole of step c alone on C0 with each REVERSED reading (drop CANCELLED, then R2, R3 or R4), so that each reading can be compared with the benchmark without the earlier steps in between.

In [ ]:
run_chain('C3 drop CANCELLED')
run_chain('C4 drop REVERSED (R1, BOOKED-only)')
reversal_options = [('R2 drop original and REVERSED rows', [reversed_R2_drop_original_too]),
                    ('R3 one offset per original', [reversed_R3_one_offset]),
                    ('R4 every REVERSED row offsets (double offset, evidence only)', [reversed_R4_every_offset])]
for label, steps in reversal_options:
    run_branch('c2', f'c2 after C3: {label}', 'C3 drop CANCELLED', steps)

run_alone('c1', 'c1 alone: drop CANCELLED', [drop_cancelled])
run_alone('c2', 'c2 alone: R1 drop REVERSED', [reversed_R1_drop_reversed])
run_alone('c', 'c alone: BOOKED only (Guide shared benchmark)', [drop_cancelled, reversed_R1_drop_reversed], keep=True)
for label, steps in reversal_options:
    run_alone('c2', f'c2 alone: {label}', steps)
for label, steps in reversal_options:
    run_alone('c', f'c alone: drop CANCELLED, then {label}', [drop_cancelled] + steps, keep=True)

step_c_names = (['C2 conflicting keys not evaluable', 'C3 drop CANCELLED', 'C4 drop REVERSED (R1, BOOKED-only)']
                + [f'c2 after C3: {label}' for label, _ in reversal_options]
                + ['c1 alone: drop CANCELLED', 'c2 alone: R1 drop REVERSED']
                + [f'c2 alone: {label}' for label, _ in reversal_options]
                + ['c alone: BOOKED only (Guide shared benchmark)']
                + [f'c alone: drop CANCELLED, then {label}' for label, _ in reversal_options])
show_step(step_c_names)

**What this tells us.**

- **Dropping CANCELLED is the largest single improvement:** +4,244 matched keys against positions (+4,222 against the custodian). The 8 nonzero rebuild-only keys disappear, so both reported tables exclude cancelled trades.
- **The REVERSED readings barely move the chain:** R1 473,944, R2 473,947, R3 473,946.
- **R4, the double offset, adds 681 matches (474,625).** Applied alone on C0 after dropping CANCELLED, R4 reaches 476,699 matches with only 937 differences (rate A 99.80%), the best fit of any variant.
- **The book of record therefore seems to treat each REVERSED row as an offset**, which reverses every original twice. That is the reading the Guide warns against, so this is evidence of a book policy that looks wrong, not a reason to adopt R4.
- **The Guide's shared benchmark (BOOKED only) gives 476,014 matches** (rate A 99.66%), more than chain C4 (473,944), because it keeps the duplicates that the book also counts.

### 9.5 Step d: negative raw quantities

The Guide says quantity is meant to be a nonnegative magnitude and that negative raw quantities **must be investigated, not silently turned into absolute values**. So first we look at them: which transaction type and status they have, whether they belong to the duplicate pair whose quantity flipped sign (section 5), whether `gross_amount` is negative too, and which sign of the quantity reproduces `gross_amount` as `quantity x price` (within the gross tolerance from config).

In [ ]:
negative = trades['quantity_number'] < 0
negative_rows = trades[negative]
sign_flip_ids = conflicts.loc[conflicts['kind'] == 'sign flip', 'transaction_id']
negative_gross = pd.to_numeric(negative_rows['gross_amount'], errors='coerce')
negative_price = pd.to_numeric(negative_rows['price'], errors='coerce')

print(f'rows with a negative raw quantity: {len(negative_rows):,} | keys touched: '
      f'{negative_rows[KEY].drop_duplicates().shape[0]:,}')
display(pd.crosstab(negative_rows['transaction_type'], negative_rows['status'], margins=True, margins_name='total'))
print('pair type of the negative rows:', negative_rows['pair_type'].value_counts().to_dict())
print('negative rows in the sign-flip conflict pair(s):', int(negative_rows['transaction_id'].isin(sign_flip_ids).sum()))
print('gross_amount of the negative rows: negative', int((negative_gross < 0).sum()),
      '| zero', int((negative_gross == 0).sum()), '| positive', int((negative_gross > 0).sum()),
      '| not a number', int(negative_gross.isna().sum()))
fits_raw = (negative_gross - negative_rows['quantity_number'] * negative_price).abs() <= GROSS_TOLERANCE
fits_absolute = (negative_gross - negative_rows['quantity_number'].abs() * negative_price).abs() <= GROSS_TOLERANCE
print(f'gross_amount = quantity x price with the raw negative quantity: {int(fits_raw.sum())} | '
      f'with its absolute value: {int(fits_absolute.sum())} | neither: {int((~fits_raw & ~fits_absolute).sum())}')

**What this tells us.** All 655 negative quantities are **BUY** rows (652 BOOKED, 2 CANCELLED, 1 REVERSED), each on a different key. One of them is the sign-flip copy found in section 5. Their `gross_amount` is always positive, and in 654 of 655 rows it equals the **absolute** quantity times the price (0 with the raw negative, 1 with neither). So the other fields say the trade size is positive and only the sign of `quantity` is wrong. This is evidence for a sign error, but it is still a repair we have not agreed on.

The options for step d: keep the raw negative value (the chain, C5: nothing changes, the affected keys are counted), mark those keys **not evaluable**, or use the absolute value. The absolute value is a repair the Guide forbids as a silent default, so it appears only as a labelled sensitivity and is never carried forward.

In [ ]:
run_chain('C5 negatives kept raw')
negative_options = [('negatives not evaluable', [negatives_not_evaluable]),
                    ('negatives as absolute value (sensitivity)', [negatives_as_absolute])]
for label, steps in negative_options:
    run_branch('d', f'd after C4: {label}', 'C4 drop REVERSED (R1, BOOKED-only)', steps)
for label, steps in negative_options:
    run_alone('d', f'd alone: {label}', steps)

show_step(['C4 drop REVERSED (R1, BOOKED-only)', 'C5 negatives kept raw']
          + [f'd after C4: {label}' for label, _ in negative_options]
          + [f'd alone: {label}' for label, _ in negative_options])

**What this tells us.** Keeping the raw value changes nothing (C5 = C4). Using the absolute value would turn 638 matched keys into differences against positions, and marking the keys not evaluable moves 645 keys into that group. So **the book carries the raw negative quantity**: it subtracts the units of a BUY. The positions on those keys are therefore probably wrong in the book as well. The chain keeps the raw value and lists the keys; the absolute value stays a sensitivity.

### 9.6 Step e: blank quantities

A blank quantity has no number, so by our definition its key is **not evaluable** in every variant so far. Step e makes that explicit. First we count the blank rows and the keys they touch, then we show the one alternative, counting a blank as zero units, as a labelled sensitivity. Zero is a guess, not a fact, so it is never carried forward.

In [ ]:
blank_rows = trades[trades['quantity_blank']]
print(f'rows with a blank quantity: {len(blank_rows):,} | keys touched: {blank_rows[KEY].drop_duplicates().shape[0]:,}')
display(pd.crosstab(blank_rows['transaction_type'], blank_rows['status'], margins=True, margins_name='total'))
print('pair type of the blank rows:', blank_rows['pair_type'].value_counts().to_dict())
print('blank rows whose gross_amount and price are both filled:',
      int(((blank_rows['gross_amount'] != '') & (blank_rows['price'] != '')).sum()))
chain_c4_rows = apply_steps(trades, chain_steps['C4 drop REVERSED (R1, BOOKED-only)'])
print(f"blank rows still in the chain after C4: {int(chain_c4_rows['quantity_blank'].sum()):,} | keys touched: "
      f"{chain_c4_rows.loc[chain_c4_rows['quantity_blank'], KEY].drop_duplicates().shape[0]:,}")

run_chain('C6 blanks not evaluable')
run_branch('e', 'e after C5: blanks as zero (sensitivity)', 'C5 negatives kept raw', [blanks_as_zero])
run_alone('e', 'e alone: blanks as zero (sensitivity)', [blanks_as_zero])
show_step(['C5 negatives kept raw', 'C6 blanks not evaluable', 'e after C5: blanks as zero (sensitivity)',
           'e alone: blanks as zero (sensitivity)'])

**What this tells us.** 2,819 rows have a blank quantity (1,579 BUY, 1,240 SELL; 2,806 BOOKED), touching 2,810 keys; 2,797 keys still carry one after C4. 2,813 of the blank rows have both a price and a gross amount, so the quantity could perhaps be derived from `gross_amount / price`. We have not done that: it would be a repair.

Making blanks explicit changes nothing on the chain (C6 = C5), as expected. **Counting a blank as zero lifts the matches by 2,611 (to 476,555)** and empties the not-evaluable groups, except for 45 keys that are also conflicting duplicates. The 157 rebuild-only keys then all net to zero: they were absent from positions because, in the book, a blank counts as nothing. So the book appears to treat a missing quantity as zero, which silently drops units the trade may well have had.

### 9.7 Step f: the as-of cut-off (trade date and settlement date)

The snapshot is 10 July 2026 (from config). The **trade-date version** keeps trades with `trade_date` on or before that date. The **settlement-date version** keeps trades with a scheduled `settlement_date` on or before it. Scheduled is not the same as settled: the Guide says the date is a plan, not proof. A trade with a blank `settlement_date` cannot be placed, so its key is **not evaluable** in the settlement version. Trades that settle before their trade date are kept as dated, and counted.

In [ ]:
def cutoff_counts(rows):
    return {'rows': len(rows),
            'trade_date after the cut-off': int((rows['trade_day'] > CUTOFF).sum()),
            'trade_date not readable': int(rows['trade_day'].isna().sum()),
            'settlement_date after the cut-off': int((rows['settlement_day'] > CUTOFF).sum()),
            'settlement_date blank or not readable': int(rows['settlement_day'].isna().sum()),
            'settlement_date before trade_date': int((rows['settlement_day'] < rows['trade_day']).sum())}

chain_c6_rows = apply_steps(trades, chain_steps['C6 blanks not evaluable'])
display(pd.DataFrame({'all rows (C0)': cutoff_counts(trades), 'chain rows after C6': cutoff_counts(chain_c6_rows)}))

run_chain('C7 trade-date cut-off')
run_chain('C7s settlement-date cut-off')
run_alone('f', 'f alone: trade-date cut-off', [trade_date_cutoff])
run_alone('f', 'f alone: settlement-date cut-off', [settlement_date_cutoff])
show_step(['C6 blanks not evaluable', 'C7 trade-date cut-off', 'C7s settlement-date cut-off',
           'f alone: trade-date cut-off', 'f alone: settlement-date cut-off'])

**What this tells us.** No trade is dated after 10 July, so the trade-date cut-off changes nothing (C7 = C6). The settlement-date version is very different. 64,065 trades settle after the snapshot (63,763 on the chain), 1,687 have no settlement date, and 1,124 settle before their trade date. Against positions, the matches fall by 59,970 to 413,974 (rate A 87.70%), 4,062 more keys appear only in positions (all their trades settle later), and 1,530 more keys become not evaluable because of a blank settlement date. **`positions` is clearly a trade-date book.** The settlement-date version is a sensitivity of what is scheduled to have settled, not a second book to reconcile to.

### 9.8 The whole chain, step by step

The table below puts the chain together, so the effect of each step can be read from one row to the next. The second table shows the **change** in each group made by that step (the settlement-date version is compared with C6, the step it starts from). The full table of every variant, against both reported tables, is saved to `outputs/01_rebuild_steps.csv`.

In [ ]:
chain_names = list(chain_steps)
previous_in_chain = dict(zip(chain_names[1:], chain_names[:-1]))
previous_in_chain['C7s settlement-date cut-off'] = 'C6 blanks not evaluable'
change_columns = GROUPS + ['matched exactly', 'keys in union']

for reported_name in reported_tables:
    chain_view = step_table(chain_names, reported_name)
    print(f'Chain against {reported_name}:')
    display(chain_view)
    changes = pd.DataFrame({name: chain_view.loc[name, change_columns] - chain_view.loc[before, change_columns]
                            for name, before in previous_in_chain.items()}).T
    print(f'Change made by each step, against {reported_name}:')
    display(changes)

long_lines = []
for name, record in step_results.items():
    for reported_name in reported_tables:
        long_lines.append({'step': record['step'], 'variant': name, 'kind': record['kind'],
                           'compared with': reported_name, 'rows used': record['rows used'],
                           'keys rebuilt': record['keys rebuilt'],
                           'keys with a negative quantity': record['keys with a negative quantity'],
                           'keys with a blank quantity': record['keys with a blank quantity'],
                           **record[reported_name],
                           'transaction_count equal': (record['transaction_count equal']
                                                       if reported_name == 'positions' else '')})
rebuild_steps = pd.DataFrame(long_lines)
rebuild_steps.to_csv(OUTPUTS / '01_rebuild_steps.csv', index=False)
print(f'saved {len(rebuild_steps)} lines ({len(step_results)} variants x {len(reported_tables)} tables) '
      f'to outputs/01_rebuild_steps.csv')

**What this tells us.** Read down the chain against positions:

| Step | Effect |
|---|---|
| a, dedup | −2,009 matched (the book counts duplicates) |
| b, conflicts | 39 keys become not evaluable |
| c1, CANCELLED | +4,244 matched, the biggest gain |
| c2, R1 | almost no change |
| d, e, f (trade date) | no change, by construction |
| f, settlement date | about −60,000 matched |

The chain ends at 473,944 matched, 3,656 different, 384 only in positions, 42 netting to zero, and 2,836 not evaluable, out of 480,862 keys (rate A 99.23%, rate B 98.56%). Against the custodian the steps move the same way, ending at 471,583 matched (rate A 98.84%). The full table of all 37 variants is in `outputs/01_rebuild_steps.csv`.

### 9.9 `transaction_count` as an extra check

The dictionary describes `positions.transaction_count` as the *"reported number of contributing transactions; validate against a stated inclusion policy"*. Each variant is an inclusion policy, so for each one we count the keys (in both the rebuild and positions) where the number of rows that contribute to the rebuild equals `transaction_count`. Under R3 and R4 the REVERSED rows that offset count as contributing. A policy that fits on every key tells us how the book counts, which is not the same as how it *should* count.

In [ ]:
count_fit = pd.DataFrame([{'variant': name, 'kind': record['kind'], 'keys in both': record['keys in both'],
                           'transaction_count equal': record['transaction_count equal'],
                           'share equal': round(record['transaction_count equal'] / record['keys in both'], 6)}
                          for name, record in step_results.items()])
count_fit = count_fit.sort_values(['transaction_count equal', 'variant'], ascending=[False, True]).set_index('variant')
count_fit

**What this tells us.** Two inclusion policies fit `transaction_count` on **every one of the 480,279 keys**. One counts every row except CANCELLED; the other also lets the REVERSED rows offset (the count is the same because R4 keeps those rows). Both mean: BOOKED and REVERSED rows, with duplicate copies counted twice. The Guide's BOOKED-only benchmark misses exactly the 699 keys with reversals (479,580). This is independent evidence, from a different field, for what the quantities already showed: **the book counts each duplicate copy and each REVERSED row as a contributing transaction.**

### 9.10 The 480 positions the custodian does not report

Section 6 found 480 keys in `positions` with no row in `comparison_positions`. Does the transaction history support the book's quantity for them? For each variant we count how many of the 480 the rebuild **matches** (the history supports the book), contradicts (**different**), cannot evaluate, or has **no trades** for (only in the reported table).

In [ ]:
book_only_view = pd.DataFrame({name: record['book-only 480'] for name, record in step_results.items()}).T
book_only_view = book_only_view.reindex(columns=GROUPS).fillna(0).astype(int)
book_only_view = book_only_view.loc[:, book_only_view.sum() > 0]          # hide groups that never occur
book_only_view.insert(0, 'kind', [step_results[name]['kind'] for name in book_only_view.index])
book_only_view

**What this tells us.** For the 480 positions the custodian does not report, the trade history **supports the book** in nearly every case. Under the raw benchmark 473 match, 5 differ and 2 are not evaluable; under the Guide's BOOKED-only benchmark 477 match and 1 differs; under the book's own policy (drop CANCELLED, then R4) 478 match and none differ. None of them lacks trades. So these look like holdings **missing from the custodian feed**, not holdings our book invented. Under the settlement-date version 61 of them differ and 3 lose all their trades. That fits a custodian that only reports what has settled for some of them, which we have not tested.

### 9.11 Orphan keys

The rebuild keeps every trade, also those on the 195 account_ids missing from `accounts` and the 15 security_ids missing from `securities` (section 6). Nothing is dropped through an inner join. First we count how many transactions use such IDs. Then, for each variant, we show the groups of the keys that involve an orphan account or security.

In [ ]:
orphan_trades = pd.Series({
    'transaction rows on an account missing from accounts': int((~trades['account_id'].isin(accounts['account_id'])).sum()),
    'distinct such account_ids': trades.loc[~trades['account_id'].isin(accounts['account_id']), 'account_id'].nunique(),
    '...of which also in the 195 from positions': int(pd.Series(trades.loc[~trades['account_id'].isin(
        accounts['account_id']), 'account_id'].unique()).isin(orphan_account_ids).sum()),
    'transaction rows on a security missing from securities': int((~trades['security_id'].isin(securities['security_id'])).sum()),
    'distinct such security_ids': trades.loc[~trades['security_id'].isin(securities['security_id']), 'security_id'].nunique(),
    '...of which also in the 15 from positions': int(pd.Series(trades.loc[~trades['security_id'].isin(
        securities['security_id']), 'security_id'].unique()).isin(orphan_security_ids).sum()),
})
display(orphan_trades.to_frame('count'))

orphan_view = pd.DataFrame({name: record['orphan keys'] for name, record in step_results.items()}).T
orphan_view = orphan_view.reindex(columns=GROUPS).fillna(0).astype(int)
orphan_view = orphan_view.loc[:, orphan_view.sum() > 0]
orphan_view.insert(0, 'kind', [step_results[name]['kind'] for name in orphan_view.index])
orphan_view

**What this tells us.** The transactions use exactly the same orphan IDs as positions: 706 trade rows on the 195 missing accounts, and 1,131 trade rows on the 15 missing securities (the spin-off targets). The rebuild keeps all of them. On the keys involved, the rebuild agrees with the book almost everywhere (1,809 matched and 7 different under C0; 1,813 and 3 under BOOKED-only), so the trades and positions are consistent with each other. **The problem is the master data, not the arithmetic**: the platform books and holds positions on accounts and securities it has never set up.

### 9.12 One account, traced by hand

**The rule, written before looking:** among the accounts that have at least one *different* key against `positions` under the Guide's shared benchmark (BOOKED only), take the account with the **most distinct securities** (counted over its keys in the rebuild and in positions). If several accounts tie, take the first in sorted `account_id` order.

For that account we show (1) every trade, sorted by security, trade date and transaction ID, with the flags used above and `gross_amount / price` as a second opinion on the quantity from different fields; and (2) one row per security with the rebuilt quantity under the main variants, side by side with `positions` (quantity and `transaction_count`) and `comparison_positions`.

In [ ]:
benchmark = kept_comparisons['c alone: BOOKED only (Guide shared benchmark)']
accounts_with_difference = benchmark.loc[benchmark['group'] == 'different', 'account_id'].unique()
securities_per_account = (benchmark[benchmark['account_id'].isin(accounts_with_difference)]
                          .groupby('account_id')['security_id'].nunique().rename('securities').reset_index())
ranked = securities_per_account.sort_values(['securities', 'account_id'], ascending=[False, True])
trace_account = ranked.iloc[0]['account_id']
print(f'accounts with at least one difference: {len(accounts_with_difference):,}')
print(f'traced account: {trace_account} ({ranked.iloc[0]["securities"]} securities; '
      f'tied accounts at that count: {int((ranked["securities"] == ranked.iloc[0]["securities"]).sum())})')

account_trades = trades[trades['account_id'] == trace_account]
trace_trade_table = (transactions.loc[account_trades.index, ['security_id', 'transaction_id', 'status',
                                                             'transaction_type', 'trade_date', 'settlement_date',
                                                             'quantity', 'price', 'gross_amount',
                                                             'related_transaction_id']]
                     .assign(gross_over_price=(pd.to_numeric(account_trades['gross_amount'], errors='coerce')
                                               / pd.to_numeric(account_trades['price'], errors='coerce')).round(4),
                             trade_day=account_trades['trade_day'].dt.strftime('%Y-%m-%d'),
                             pair_type=account_trades['pair_type'],
                             reversed_original=account_trades['is_reversed_original'],
                             row_number=account_trades['row_number'])
                     .sort_values(['security_id', 'trade_day', 'transaction_id']))
print(f'trades of this account: {len(trace_trade_table)}')
trace_trade_table

**What this tells us.** The rule picks one account among the 1,617 with at least one difference under BOOKED-only. It has 8 securities, no other account ties with it, and it has 17 trades, all BOOKED, with no duplicates and no reversals. So none of the status or duplicate questions applies, which makes it a clean test of the arithmetic. One trade stands out: on the security that differs (see the next table), the first BUY has a price of 10,628.20 while the other two trades in that security are near 108. Its `gross_amount / price` is 29.99 instead of the quantity 2,999, so the price looks 100 times too large: the same scale error as in the duplicate conflicts of section 5.

Now the side-by-side view for the same account. Each variant is rebuilt from this account's trades only, with exactly the same step functions as above. That is safe because every flag the steps use (copy number, first reversal) was computed on the full table. A key the variant cannot evaluate is shown as "not evaluable".

In [ ]:
TRACE_VARIANTS = {
    'C0 raw': [raw_rows],
    'BOOKED only (Guide)': [raw_rows, drop_cancelled, reversed_R1_drop_reversed],
    'C7 chain, trade date': chain_steps['C7 trade-date cut-off'],
    'C7s chain, settlement date': chain_steps['C7s settlement-date cut-off'],
    'R2 after C3': chain_steps['C3 drop CANCELLED'] + [reversed_R2_drop_original_too],
    'R3 after C3': chain_steps['C3 drop CANCELLED'] + [reversed_R3_one_offset],
    'R4 after C3 (double offset)': chain_steps['C3 drop CANCELLED'] + [reversed_R4_every_offset],
}

side_by_side = {}
for label, steps in TRACE_VARIANTS.items():
    rebuilt = rebuild(apply_steps(account_trades, steps)).set_index('security_id')
    shown = rebuilt['rebuilt_quantity'].map('{:,.4f}'.format)
    shown[rebuilt['not_evaluable']] = 'not evaluable'
    side_by_side[label] = shown
side_by_side = pd.DataFrame(side_by_side)

reported_for_account = {
    'positions quantity': positions.loc[positions['account_id'] == trace_account].set_index('security_id')['quantity'],
    'positions transaction_count': (positions.loc[positions['account_id'] == trace_account]
                                    .set_index('security_id')['transaction_count']),
    'comparison_positions quantity': (comparison_positions.loc[comparison_positions['account_id'] == trace_account]
                                      .set_index('security_id')['quantity']),
    'group under BOOKED only': benchmark.loc[benchmark['account_id'] == trace_account].set_index('security_id')['group'],
}
trace_view = side_by_side.join(pd.DataFrame(reported_for_account), how='outer').fillna('(no row)')
trace_view

**What this tells us.** Seven of the eight securities agree everywhere: every trade-date variant gives the same quantity as `positions` and the custodian, and `transaction_count` equals the number of trades. The settlement-date version differs on three of them only because a sale settles after 10 July (in one security the only trade settles later, so it has no row at all).

The eighth security is a real break. **The trades add up to 10,849 under every variant, but `positions` and `comparison_positions` both say 14,147.29** (about 1.304 times more), and `transaction_count` (3) matches the three trades. Statuses, duplicates and the cut-off cannot explain it. The custodian agrees with the book, not with the trades, so the break sits between the trade history and both reported tables.

- **Observed:** the gap of +3,298.29 units; the price on one of the three trades that is 100 times too large.
- **Not yet shown:** whether the two are linked. This is a candidate for an evidenced trace at Check-in 2.

<a id="defects"></a>
## 10. Defect log

A small table of what this notebook found, built from the variables computed above (no numbers typed by hand). The columns follow the Working Guide: what the problem is, where, how many rows, how we detected it, and the evidence status (**confirmed**, **unresolved** or **cleared**). "Confirmed" means the fact is directly observed in the data. "Unresolved" means the observation is certain but its cause or meaning is not, or it may be a convention. The table is saved to `outputs/01_defect_log.csv` so it can be copied into the defect catalog.

In [ ]:
format_only_rows = int(pair_differs_only_by_format.sum()) * 2   # two rows per pair
real_conflict_rows = int(pair_has_real_conflict.sum()) * 2
pipe_repeated_rows = len(repeated_pipe)

# Numbers for the new lines (sections 7 to 9)
benchmark_name = 'c alone: BOOKED only (Guide shared benchmark)'
book_policy_name = 'c alone: drop CANCELLED, then R4 every REVERSED row offsets (double offset, evidence only)'
benchmark_positions = step_results[benchmark_name]['positions']
book_policy_positions = step_results[book_policy_name]['positions']
reversal_keys = trades.loc[trades['status'] == 'REVERSED', KEY].drop_duplicates()
reversal_key_index = pd.MultiIndex.from_frame(reversal_keys)
r4_on_reversal_keys = kept_comparisons[book_policy_name]
r4_matched_reversal_keys = int((r4_on_reversal_keys.set_index(KEY).index.isin(reversal_key_index)
                                & (r4_on_reversal_keys['group'] == 'matched')).sum())
duplicate_keys = trades.loc[trades['pair_type'] != 'not repeated', KEY].drop_duplicates().shape[0]
dedup_effect = (step_results['C1 drop exact and format-only copies']['positions']['matched']
                - step_results['C0 raw benchmark']['positions']['matched'])
count_policies_all_keys = int((count_fit['transaction_count equal'] == count_fit['keys in both']).sum())
blank_zero_gain = (step_results['e after C5: blanks as zero (sensitivity)']['positions']['matched']
                   - step_results['C5 negatives kept raw']['positions']['matched'])
absolute_loss = (step_results['C4 drop REVERSED (R1, BOOKED-only)']['positions']['matched']
                 - step_results['d after C4: negatives as absolute value (sensitivity)']['positions']['matched'])
rebook_changed = gross_fit.loc['BOOKED -> CANCELLED']
book_only_benchmark = step_results[benchmark_name]['book-only 480']
not_expected_rows = int(field_surprises.loc[field_surprises['issue'] == 'filled where not expected', 'rows affected'].sum())
merger_rows = int(filled_by_type.loc['MERGER', 'rows'])
merger_without_target = merger_rows - int(filled_by_type.loc['MERGER', 'target_security_id filled'])

defect_log = pd.DataFrame([
    {'defect': 'Repeated transaction_id, exact full-row copy',
     'table': 'transactions', 'rows': exact_copy_rows,
     'how detected': 'transaction_id appears more than once and all 16 columns are equal',
     'evidence status': 'confirmed'},
    {'defect': 'Repeated transaction_id, copies differ only by date format',
     'table': 'transactions', 'rows': format_only_rows,
     'how detected': 'copies differ as text but are equal after normalising YYYY-MM-DD and DD/MM/YYYY',
     'evidence status': 'confirmed'},
    {'defect': 'Repeated transaction_id, copies conflict on a fact',
     'table': 'transactions', 'rows': real_conflict_rows,
     'how detected': 'copies still differ after date normalisation (blank, sign flip, currency, other)',
     'evidence status': 'unresolved (which copy is right is unknown)'},
    {'defect': 'Repeated transaction_id in the pipeline',
     'table': 'pipeline_events', 'rows': pipe_repeated_rows,
     'how detected': 'transaction_id appears more than once; same row positions as in transactions',
     'evidence status': 'confirmed'},
    {'defect': 'trade_date written as DD/MM/YYYY instead of YYYY-MM-DD',
     'table': 'transactions', 'rows': format_table.loc['slash DD/MM/YYYY', 'trade_date'],
     'how detected': 'date format count on the full table',
     'evidence status': 'confirmed (format inconsistency, readable)'},
    {'defect': 'Blank settlement_date',
     'table': 'transactions', 'rows': format_table.loc['blank', 'settlement_date'],
     'how detected': 'date format count on the full table',
     'evidence status': 'confirmed (missing value)'},
    {'defect': 'Position in our book with no custodian record',
     'table': 'positions', 'rows': int(merge_counts['only in positions']),
     'how detected': ('full outer merge on account_id, security_id, as_of_date; the trade history matches the book on '
                      f"{book_only_benchmark.get('matched', 0)} of {len(book_only)} under BOOKED-only"),
     'evidence status': 'unresolved (history supports the book: likely missing from the custodian feed)'},
    {'defect': 'Position on an account_id missing from accounts',
     'table': 'positions', 'rows': rows_with_orphan_account,
     'how detected': 'positions.account_id not found in accounts.account_id',
     'evidence status': 'confirmed'},
    {'defect': 'Position on a security_id missing from securities',
     'table': 'positions', 'rows': rows_with_orphan_security,
     'how detected': ('positions.security_id not found in securities.security_id; '
                      f'{orphan_is_target.sum()} of {len(orphan_security_ids)} are corporate action targets'),
     'evidence status': 'confirmed'},
    # ----- section 7: the four small tables
    {'defect': 'Corporate action target security missing from securities',
     'table': 'corporate_actions', 'rows': int(target_summary['events whose target is missing from securities']),
     'how detected': (f"target_security_id not in securities; {target_summary['missing targets traded and held']} "
                      f"targets traded and held ({target_summary['transaction rows on missing targets']:,} trades, "
                      f"{target_summary['position rows on missing targets']:,} positions, "
                      f"{target_summary['price rows on missing targets']} prices)"),
     'evidence status': 'confirmed'},
    {'defect': 'MERGER without a target_security_id',
     'table': 'corporate_actions', 'rows': merger_without_target,
     'how detected': f'{merger_without_target} of {merger_rows} MERGER rows have a blank target; a merger ratio needs one',
     'evidence status': 'confirmed (incomplete terms: quarantine)'},
    {'defect': 'ratio or cash_rate filled for an event type that does not use it',
     'table': 'corporate_actions', 'rows': not_expected_rows,
     'how detected': 'fields filled per action_type compared with the Guide conventions, written down before counting',
     'evidence status': 'unresolved (our reading of the Guide)'},
    {'defect': 'ex_date after effective_date',
     'table': 'corporate_actions', 'rows': ex_after_effective,
     'how detected': 'both dates normalised and compared',
     'evidence status': 'unresolved (date definitions to confirm with faculty)'},
    {'defect': 'Preferred vendor has no price on a date',
     'table': 'prices', 'rows': len(missing_vendor_dates),
     'how detected': ('vendor x date grid per security; missing: ' + ', '.join(
         f'{vendor} on {day}' for vendor, day in
         missing_vendor_dates[['vendor', 'missing date']].drop_duplicates().itertuples(index=False))),
     'evidence status': 'confirmed (missing value)'},
    {'defect': 'Held security with no price at all',
     'table': 'positions', 'rows': int(positions['security_id'].isin(held_without_price).sum()),
     'how detected': (f'{len(held_without_price)} held security_ids absent from prices; '
                      f'{int(held_without_price.isin(missing_targets).sum())} are spin-off targets'),
     'evidence status': 'confirmed'},
    {'defect': 'Price currency with no FX rate',
     'table': 'prices', 'rows': int(prices['currency'].isin(xyz_like).sum()),
     'how detected': (f"currency {', '.join(xyz_like)} not in fx_rates; {len(no_fx_securities)} securities, held in "
                      f"{int(positions['security_id'].isin(no_fx_securities).sum()):,} positions"),
     'evidence status': 'confirmed'},
    {'defect': 'USD rate to USD is not 1',
     'table': 'fx_rates', 'rows': usd_rows_not_one,
     'how detected': f"USD rate_to_usd from {usd['rate'].min():.6f} to {usd['rate'].max():.6f}",
     'evidence status': 'unresolved (error or convention: faculty question)'},
    # ----- section 8: statuses and links
    {'defect': 'CANCELLED row linked to an ID that does not exist',
     'table': 'transactions', 'rows': len(dangling),
     'how detected': 'related_transaction_id not found in transactions.transaction_id',
     'evidence status': 'confirmed'},
    {'defect': 'Two REVERSED rows for each reversed original',
     'table': 'transactions', 'rows': int(reversals_per_target.sum()),
     'how detected': (f'{len(reversals_per_target)} BOOKED originals, each referenced by '
                      f'{sorted(int(count) for count in reversals_per_target.unique())} REVERSED rows with the same sign as the original'),
     'evidence status': 'unresolved (meaning of REVERSED: faculty question)'},
    {'defect': 'Rebook changes quantity but keeps the original gross_amount',
     'table': 'transactions', 'rows': int(rebook_changed['gross_fits_target_quantity']),
     'how detected': (f"BOOKED -> CANCELLED links with a different quantity: gross_amount fits the cancelled quantity in "
                      f"{int(rebook_changed['gross_fits_target_quantity'])} of {int(rebook_changed['rows_with_changed_quantity'])}, "
                      f"its own in {int(rebook_changed['gross_fits_own_quantity'])}"),
     'evidence status': 'unresolved (which quantity is right is unknown)'},
    # ----- section 9: reconstruction
    {'defect': 'Negative trade quantity',
     'table': 'transactions', 'rows': len(negative_rows),
     'how detected': (f"quantity < 0, all {', '.join(sorted(negative_rows['transaction_type'].unique()))}; gross_amount "
                      f"fits the absolute value in {int(fits_absolute.sum())}; the book keeps the raw sign "
                      f"(absolute values would lose {absolute_loss} matches)"),
     'evidence status': 'confirmed (sign contradicts gross_amount)'},
    {'defect': 'Blank trade quantity',
     'table': 'transactions', 'rows': len(blank_rows),
     'how detected': (f'quantity is blank; the book appears to count it as zero '
                      f'(blank-as-zero adds {blank_zero_gain:,} matches)'),
     'evidence status': 'confirmed (missing value)'},
    {'defect': 'settlement_date before trade_date',
     'table': 'transactions', 'rows': cutoff_counts(trades)['settlement_date before trade_date'],
     'how detected': 'both dates normalised and compared',
     'evidence status': 'confirmed'},
    {'defect': 'Book counts each duplicate copy as a trade',
     'table': 'positions', 'rows': duplicate_keys,
     'how detected': (f'removing duplicate copies changes the matches by {dedup_effect:,}; transaction_count fits '
                      'only policies that count both copies'),
     'evidence status': 'unresolved (book policy, consistent with section 5)'},
    {'defect': 'Book applies every REVERSED row as an offset (double offset)',
     'table': 'positions', 'rows': len(reversal_keys),
     'how detected': (f'on {r4_matched_reversal_keys} of {len(reversal_keys)} keys with a reversal the book equals '
                      'BOOKED minus each REVERSED row; transaction_count counts the REVERSED rows'),
     'evidence status': 'unresolved (meaning of REVERSED; the Guide warns against double offsets)'},
    {'defect': 'Position with no contributing trade',
     'table': 'positions', 'rows': benchmark_positions['only in reported table'],
     'how detected': 'key in positions but in no rebuild variant; transaction_count is 0',
     'evidence status': 'unresolved (no opening balance is provided)'},
    {'defect': 'Rebuilt quantity differs from the book (BOOKED-only benchmark)',
     'table': 'positions', 'rows': benchmark_positions['different'],
     'how detected': (f"full outer comparison, tolerance {TOLERANCE}; {book_policy_positions['different']} remain "
                      "under the book's apparent policy"),
     'evidence status': 'unresolved (breaks to classify)'},
])

OUTPUTS.mkdir(exist_ok=True)
defect_log.to_csv(OUTPUTS / '01_defect_log.csv', index=False)
defect_log

**What this tells us.** The log now has 27 lines, all computed above: 16 confirmed and 11 unresolved (no flag is cleared yet). The new lines from sections 7 to 9 fall into three groups:

- **Master and reference data:** missing spin-off targets, mergers without a target, a missing preferred price on the snapshot date, the XYZ currency, the USD rate.
- **Transaction records:** dangling cancellations, two REVERSED rows per original, rebooks that keep the old gross amount, negative and blank quantities.
- **How the book of record seems to be built:** duplicates counted twice, every REVERSED row applied as an offset, blanks counted as zero.

The last group is unresolved on purpose. It describes the book's behaviour, and whether that behaviour is wrong depends on the meaning of REVERSED, which is still open. The file is in `outputs/01_defect_log.csv`.

<a id="verification"></a>
## 11. Verification and cross-check

### 11.1 Second method: DuckDB

To make sure the pandas numbers are not an artefact of how we wrote the code, we compute some of them a second, independent way with DuckDB (SQL run directly on the `.csv.gz` files). As the repo rules require, DuckDB reads every column as text (`all_varchar=true`). If both methods agree, we can trust the number more.

In [ ]:
con = duckdb.connect()   # an in-memory database: nothing is written to disk

def duck_source(name):
    """SQL snippet that reads one table as text."""
    return f"read_csv('{table_path(name).as_posix()}', all_varchar=true, header=true)"

duck_distinct_ids = con.sql(f"""
    SELECT COUNT(DISTINCT transaction_id) FROM {duck_source('transactions')}
""").fetchone()[0]

duck_repeated_ids = con.sql(f"""
    SELECT COUNT(*) FROM (
        SELECT transaction_id FROM {duck_source('transactions')}
        GROUP BY transaction_id HAVING COUNT(*) > 1
    )
""").fetchone()[0]

duck_merge = con.sql(f"""
    SELECT CASE WHEN c.account_id IS NULL THEN 'only in positions'
                WHEN p.account_id IS NULL THEN 'only in comparison_positions'
                ELSE 'in both' END AS side,
           COUNT(*) AS n
    FROM {duck_source('positions')} AS p
    FULL OUTER JOIN {duck_source('comparison_positions')} AS c
      ON p.account_id = c.account_id
     AND p.security_id = c.security_id
     AND p.as_of_date = c.as_of_date
    GROUP BY side
""").df().set_index('side')['n']

pandas_vs_duckdb = pd.DataFrame([
    {'check': 'distinct transaction_ids', 'pandas': key_check.loc['distinct IDs', 'transactions'],
     'duckdb': duck_distinct_ids},
    {'check': 'repeated transaction_ids', 'pandas': key_check.loc['IDs that repeat', 'transactions'],
     'duckdb': duck_repeated_ids},
] + [
    {'check': f'positions merge: {side}', 'pandas': merge_counts[side],
     'duckdb': int(duck_merge.get(side, 0))}
    for side in side_names.values()
])
pandas_vs_duckdb['agree?'] = (pandas_vs_duckdb['pandas'] == pandas_vs_duckdb['duckdb']).map({True: 'yes', False: 'NO'})
pandas_vs_duckdb

**What this tells us.** DuckDB, reading the raw files with SQL, gives exactly the same five numbers as pandas: 1,407,428 distinct IDs, 2,105 repeated IDs, and 480,183 / 480 / 0 for the positions merge. The results do not depend on how we wrote the pandas code.

**The reconstruction, a second way.** Now the main numbers of section 9: the seven groups and both match rates, for the raw benchmark and for every step of the chain (C0 to C7, plus the settlement-date version C7s), against both reported tables. The SQL is written independently of the pandas code:

- **Blanks.** DuckDB reads an empty field as `NULL` (not as an empty string), so "blank" is tested with `IS NULL`.
- **Dates.** They are parsed with `COALESCE(TRY_STRPTIME(d, '%Y-%m-%d'), TRY_STRPTIME(d, '%d/%m/%Y'))`; a value in neither format stays `NULL` and marks its key.
- **Duplicates.** They are removed with `SELECT DISTINCT` over all columns after date normalisation. That drops exact and format-only copies without using row order, and any ID still appearing twice is a conflicting pair (marked not evaluable).
- **Settings.** Quantities are summed as `DOUBLE`, and the same tolerance and cut-off from `config/exploration.toml` are passed as query parameters.

In [ ]:
def duck_day(column):
    """SQL that reads a YYYY-MM-DD or DD/MM/YYYY text column as a DATE (NULL if neither)."""
    return f"COALESCE(TRY_STRPTIME({column}, '%Y-%m-%d'), TRY_STRPTIME({column}, '%d/%m/%Y'))::DATE"

con.execute(f"""
    CREATE OR REPLACE TEMP TABLE duck_trades AS
    SELECT *,
           {duck_day('trade_date')} AS trade_day,
           {duck_day('settlement_date')} AS settlement_day,
           TRY_CAST(quantity AS DOUBLE) AS quantity_number,
           CASE transaction_type WHEN 'BUY' THEN 1 WHEN 'SELL' THEN -1 END AS direction
    FROM {duck_source('transactions')}
""")
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE duck_dedup AS
    SELECT DISTINCT transaction_id, account_id, security_id, cusip, transaction_type,
           COALESCE(strftime(trade_day, '%Y-%m-%d'), trade_date) AS trade_date,
           COALESCE(strftime(settlement_day, '%Y-%m-%d'), settlement_date) AS settlement_date,
           quantity, price, currency, gross_amount, status, related_transaction_id, source_system,
           COALESCE(strftime({duck_day('substr(event_timestamp, 1, 10)')}, '%Y-%m-%d'), substr(event_timestamp, 1, 10))
               || substr(event_timestamp, 11) AS event_timestamp,
           settlement_note, trade_day, settlement_day, quantity_number, direction
    FROM duck_trades
""")
con.execute("""
    CREATE OR REPLACE TEMP TABLE duck_conflict_ids AS
    SELECT transaction_id FROM duck_dedup GROUP BY transaction_id HAVING COUNT(*) > 1
""")
print('rows:', con.sql('SELECT COUNT(*) FROM duck_trades').fetchone()[0],
      '| after DISTINCT:', con.sql('SELECT COUNT(*) FROM duck_dedup').fetchone()[0],
      '| conflicting IDs:', con.sql('SELECT COUNT(*) FROM duck_conflict_ids').fetchone()[0])

conflict_mark = 'transaction_id IN (SELECT transaction_id FROM duck_conflict_ids)'
DUCK_CHAIN = {   # variant -> (FROM table, WHERE condition, extra "not evaluable" condition)
    'C0 raw benchmark': ('duck_trades', 'TRUE', 'FALSE'),
    'C1 drop exact and format-only copies': ('duck_dedup', 'TRUE', 'FALSE'),
    'C2 conflicting keys not evaluable': ('duck_dedup', 'TRUE', conflict_mark),
    'C3 drop CANCELLED': ('duck_dedup', "status IS DISTINCT FROM 'CANCELLED'", conflict_mark),
}
after_c4 = "status IS DISTINCT FROM 'CANCELLED' AND status IS DISTINCT FROM 'REVERSED'"
for name in ['C4 drop REVERSED (R1, BOOKED-only)', 'C5 negatives kept raw', 'C6 blanks not evaluable']:
    DUCK_CHAIN[name] = ('duck_dedup', after_c4, conflict_mark)
DUCK_CHAIN['C7 trade-date cut-off'] = ('duck_dedup', f'{after_c4} AND (trade_day IS NULL OR trade_day <= $cutoff_day)',
                                       f'{conflict_mark} OR trade_day IS NULL')
DUCK_CHAIN['C7s settlement-date cut-off'] = ('duck_dedup',
                                             f'{after_c4} AND (settlement_day IS NULL OR settlement_day <= $cutoff_day)',
                                             f'{conflict_mark} OR settlement_day IS NULL')


def duck_groups(source, condition, mark, reported_name):
    """Group counts of one chain variant against one reported table, computed in SQL."""
    sql = f"""
        WITH rebuilt AS (
            SELECT COALESCE(account_id, '') AS account_id, COALESCE(security_id, '') AS security_id,
                   $snapshot_text AS as_of_date, TRUE AS present,
                   SUM(direction * quantity_number) AS rebuilt_quantity,
                   BOOL_OR(direction IS NULL OR quantity_number IS NULL OR ({mark})) AS not_evaluable
            FROM {source} WHERE {condition}
            GROUP BY ALL),
        reported AS (
            SELECT COALESCE(account_id, '') AS account_id, COALESCE(security_id, '') AS security_id,
                   COALESCE(as_of_date, '') AS as_of_date, TRUE AS present,
                   TRY_CAST(quantity AS DOUBLE) AS reported_quantity
            FROM {duck_source(reported_name)})
        SELECT CASE
                 WHEN b.present IS NULL THEN 'only in reported table'
                 WHEN p.present IS NULL AND b.not_evaluable THEN 'not evaluable (only in rebuild)'
                 WHEN p.present IS NULL AND ABS(b.rebuilt_quantity) <= $tolerance THEN 'only in rebuild, nets to zero'
                 WHEN p.present IS NULL THEN 'only in rebuild, nonzero'
                 WHEN b.not_evaluable OR p.reported_quantity IS NULL THEN 'not evaluable (in both)'
                 WHEN ABS(b.rebuilt_quantity - p.reported_quantity) <= $tolerance THEN 'matched'
                 ELSE 'different' END AS grp,
               COUNT(*) AS n
        FROM rebuilt AS b
        FULL OUTER JOIN reported AS p
          ON b.account_id = p.account_id
         AND b.security_id = p.security_id
         AND b.as_of_date = p.as_of_date        -- the rebuild carries the snapshot date, like the pandas key
        GROUP BY grp"""
    parameters = {'tolerance': TOLERANCE, 'snapshot_text': AS_OF_DATE}
    if '$cutoff_day' in sql:                       # only the cut-off variants use the date
        parameters['cutoff_day'] = CUTOFF.date()
    return dict(con.execute(sql, parameters).fetchall())


rebuild_check_rows = []
for name, (source, condition, mark) in DUCK_CHAIN.items():
    for reported_name in reported_tables:
        duck_counts = duck_groups(source, condition, mark, reported_name)
        pandas_summary = step_results[name][reported_name]
        duck_union = sum(duck_counts.values())
        duck_comparable = duck_counts.get('matched', 0) + duck_counts.get('different', 0)
        duck_values = {group: duck_counts.get(group, 0) for group in GROUPS}
        duck_values['keys in union'] = duck_union
        duck_values[RATE_A] = round(duck_counts.get('matched', 0) / duck_comparable, 6)
        duck_values[RATE_B] = round(duck_counts.get('matched', 0) / duck_union, 6)
        for measure, duck_value in duck_values.items():
            rebuild_check_rows.append({'variant': name, 'compared with': reported_name, 'measure': measure,
                                       'pandas': pandas_summary[measure], 'duckdb': duck_value})
rebuild_check = pd.DataFrame(rebuild_check_rows)
rebuild_check['agree?'] = (rebuild_check['pandas'] == rebuild_check['duckdb']).map({True: 'yes', False: 'NO'})
print(f"rebuild numbers compared: {len(rebuild_check)} | agree: {(rebuild_check['agree?'] == 'yes').sum()} | "
      f"disagree: {(rebuild_check['agree?'] == 'NO').sum()}")
display(rebuild_check[rebuild_check['agree?'] == 'NO'])
rebuild_check.pivot_table(index=['variant', 'compared with'], columns='measure', values='duckdb',
                          aggfunc='first', sort=False)[GROUPS + ['keys in union', RATE_A, RATE_B]]

**What this tells us.** DuckDB reproduces every reconstruction number: **180 of 180 agree** (9 variants x 2 reported tables x 10 measures), with no disagreement. The SQL deduplication by `SELECT DISTINCT` leaves 1,407,473 rows and 45 conflicting IDs, the same as the pandas chain at C1 and the 45 conflicting pairs of section 5, although it never uses row order. So the step-by-step results of section 9 do not depend on how the pandas code was written. That covers the treatment of blanks, the date parsing and the duplicate handling.

### 11.2 Cross-check against our manual results

Before this notebook, we worked some numbers out by hand: first in a chat session, and later (for sections 7 to 9) from PROJECT_SUMMARY section 5 and a quick DuckDB check. They are typed into the dictionary below **only** so we can compare; nothing above uses them. The table shows each check, the manual result, this notebook's result and whether they match. Where a number was only an estimate, the last column says so. New numbers from sections 7 to 9 that we never worked out by hand are listed as "no manual result", so they are visible but not scored. A mismatch is not something to hide or fix: it is shown, and explained in the cell after the table. It is saved to `outputs/01_cross_check.csv`.

In [ ]:
# Results worked out by hand before this notebook: a second opinion only, NOT used anywhere above.
MANUAL_RESULTS = {
    'rows: transactions': 1_409_533,
    'rows: pipeline_events': 1_409_533,
    'rows: positions': 480_663,
    'rows: comparison_positions': 480_183,
    'rows: accounts': 100_000,
    'rows: securities': 6_012,
    'rows: corporate_actions': 200,
    'rows: prices': 539_970,
    'rows: fx_rates': 180,
    'rows: ledger_nav': 2_000,
    'status BOOKED': 1_403_784,
    'status CANCELLED': 4_347,
    'status REVERSED': 1_402,
    'distinct transaction_ids': 1_407_428,
    'repeated transaction_ids': 2_105,
    'each repeated ID appears exactly twice': 'yes',
    'copies sit in adjacent rows': 'yes',
    'same row positions in both tables': 'yes',
    'every ID in one table exists in the other': 'yes',
    'rows that are exact full-row copies': 4_064,
    'trade_date conflicts after normalising': 0,
    'event_timestamp conflicts after normalising': 0,
    'pairs with a real conflict': 45,
    'conflicts: settlement_date': 21,
    'conflicts: price': 14,
    'conflicts: quantity': 6,
    'conflicts: currency': 4,
    'conflicts: settlement_note': 3,
    'repeated rows: BOOKED': 4_206,
    'repeated rows: CANCELLED': 4,
    'pipeline pairs differ: published_timestamp': 1_987,
    'pipeline pairs differ: booked_timestamp': 20,
    'pipeline pairs differ: received_timestamp': 7,
    'pipeline pairs differ: source_system': 0,
    'pipeline pairs differ: batch_id': 0,
    'pipeline pairs differ: event_timestamp': 0,
    'pipeline pairs differ: cutoff_timestamp': 0,
    'positions as_of_date values': '2026-07-10',
    'comparison_positions as_of_date values': '2026-07-10',
    'positions distinct account_ids': 100_195,
    'comparison_positions distinct account_ids': 100_195,
    'positions distinct security_ids': 6_015,
    'comparison_positions distinct security_ids': 6_015,
    'merge: in both': 480_183,
    'merge: only in positions': 480,
    'merge: only in comparison_positions': 0,
    'book-only rows with zero quantity': 0,
    'book-only quantity min': -12_322,
    'book-only quantity max': 17_690,
    'book-only quantity median (about)': 1_231.5,
    'book-only quantity mean (about)': 870.26,
    # Sections 7 to 9: from PROJECT_SUMMARY section 5 and a quick DuckDB check
    'corporate_actions: SPINOFF rows': 23,
    'corporate_actions: SPINOFF rows with target_security_id': 23,
    'corporate_actions: SPINOFF targets missing from securities': 23,
    'missing spin-off targets traded and held': 15,
    'missing spin-off targets: transaction rows': 1_131,
    'missing spin-off targets: position rows': 1_118,
    'missing spin-off targets: price rows': 0,
    'corporate_actions: MERGER rows': 31,
    'corporate_actions: MERGER rows with target_security_id': 0,
    'corporate_actions: ex_date after effective_date': 87,
    'corporate_actions: rows with a cash_rate': 200,
    'prices: vendors': 3,
    'prices: dates': 30,
    'prices: distinct securities': 6_000,
    'fx_rates: currencies': 6,
    'fx_rates: dates': 30,
    'fx_rates: USD rows exactly 1': 0,
    'fx_rates: USD lowest (about)': 0.990,
    'fx_rates: USD highest (about)': 1.010,
    'ledger_nav: funds': 500,
    'ledger_nav: share classes': 4,
    'ledger_nav: dates': 1,
    'REVERSED rows with related_transaction_id': 1_402,
    'REVERSED rows pointing to a BOOKED record': 1_402,
    'BOOKED rows with related_transaction_id': 3_511,
    'BOOKED rows pointing to a CANCELLED record': 3_511,
    'CANCELLED rows with related_transaction_id': 842,
    'CANCELLED rows pointing to an ID not in transactions': 842,
    'quantity negative': 655,
    'quantity negative on BUY rows': 655,
    'quantity blank': 2_819,
    'quantity blank on BUY rows': 1_579,
    'quantity blank on SELL rows': 1_240,
    'trade_date after 2026-07-10': 0,
    'settlement_date after 2026-07-10': 64_065,
    'settlement_date before trade_date': 1_124,
}
# Numbers we only estimated: shown for interest, not scored as match or mismatch.
MANUAL_ESTIMATES = {
    'pairs differing only by date format': 'about 28',
    'account_ids in positions missing from accounts': 'at least 195',
    'security_ids in positions missing from securities': 'at least 3',
}

The next cell collects this notebook's own result for each of those checks, from the variables computed above, and builds the comparison table. Means and medians are rounded to 2 decimals before comparing, and the USD rate to 3 decimals, because the manual values were given as "about".

In [ ]:
def yes_no(condition):
    return 'yes' if condition else 'no'

NOTEBOOK_RESULTS = {f'rows: {name}': len(tables[name]) for name in TABLE_NAMES}
NOTEBOOK_RESULTS.update({
    'status BOOKED': int(status_overall.get('BOOKED', 0)),
    'status CANCELLED': int(status_overall.get('CANCELLED', 0)),
    'status REVERSED': int(status_overall.get('REVERSED', 0)),
    'distinct transaction_ids': int(key_check.loc['distinct IDs', 'transactions']),
    'repeated transaction_ids': int(key_check.loc['IDs that repeat', 'transactions']),
    'each repeated ID appears exactly twice': yes_no((copies_per_id == 2).all()),
    'copies sit in adjacent rows': yes_no((row_gap == 1).all()),
    'same row positions in both tables': yes_no(same_positions and same_ids_at_positions),
    'every ID in one table exists in the other': yes_no(same_id_sets),
    'rows that are exact full-row copies': exact_copy_rows,
    'trade_date conflicts after normalising': int(conflicts_per_column['trade_date']),
    'event_timestamp conflicts after normalising': int(conflicts_per_column['event_timestamp']),
    'pairs with a real conflict': int(pair_has_real_conflict.sum()),
    'conflicts: settlement_date': int(conflicts_per_column['settlement_date']),
    'conflicts: price': int(conflicts_per_column['price']),
    'conflicts: quantity': int(conflicts_per_column['quantity']),
    'conflicts: currency': int(conflicts_per_column['currency']),
    'conflicts: settlement_note': int(conflicts_per_column['settlement_note']),
    'repeated rows: BOOKED': int(status_repeated.get('BOOKED', 0)),
    'repeated rows: CANCELLED': int(status_repeated.get('CANCELLED', 0)),
})
NOTEBOOK_RESULTS.update({f'pipeline pairs differ: {column}': int(pipe_differs_per_column[column])
                         for column in pipe_columns})
NOTEBOOK_RESULTS.update({
    'positions as_of_date values': basics['positions']['as_of_date values'],
    'comparison_positions as_of_date values': basics['comparison_positions']['as_of_date values'],
    'positions distinct account_ids': basics['positions']['distinct account_ids'],
    'comparison_positions distinct account_ids': basics['comparison_positions']['distinct account_ids'],
    'positions distinct security_ids': basics['positions']['distinct security_ids'],
    'comparison_positions distinct security_ids': basics['comparison_positions']['distinct security_ids'],
    'merge: in both': int(merge_counts['in both']),
    'merge: only in positions': int(merge_counts['only in positions']),
    'merge: only in comparison_positions': int(merge_counts['only in comparison_positions']),
    'book-only rows with zero quantity': int((book_only['quantity_number'] == 0).sum()),
    'book-only quantity min': book_only_summary['min'],
    'book-only quantity max': book_only_summary['max'],
    'book-only quantity median (about)': round(book_only_summary['50%'], 2),
    'book-only quantity mean (about)': round(book_only_summary['mean'], 2),
})
cutoff_all_rows = cutoff_counts(trades)
NOTEBOOK_RESULTS.update({
    'corporate_actions: SPINOFF rows': int(filled_by_type.loc['SPINOFF', 'rows']),
    'corporate_actions: SPINOFF rows with target_security_id': int(filled_by_type.loc['SPINOFF',
                                                                                      'target_security_id filled']),
    'corporate_actions: SPINOFF targets missing from securities': int(target_summary['...of which SPINOFF']),
    'missing spin-off targets traded and held': int(target_summary['missing targets traded and held']),
    'missing spin-off targets: transaction rows': int(target_summary['transaction rows on missing targets']),
    'missing spin-off targets: position rows': int(target_summary['position rows on missing targets']),
    'missing spin-off targets: price rows': int(target_summary['price rows on missing targets']),
    'corporate_actions: MERGER rows': merger_rows,
    'corporate_actions: MERGER rows with target_security_id': merger_rows - merger_without_target,
    'corporate_actions: ex_date after effective_date': ex_after_effective,
    'corporate_actions: rows with a cash_rate': int((corporate_actions['cash_rate'] != '').sum()),
    'prices: vendors': n_vendors,
    'prices: dates': n_price_dates,
    'prices: distinct securities': prices['security_id'].nunique(),
    'fx_rates: currencies': fx['currency'].nunique(),
    'fx_rates: dates': fx['rate_day'].nunique(),
    'fx_rates: USD rows exactly 1': int(usd_summary['rows exactly 1']),
    'fx_rates: USD lowest (about)': round(usd['rate'].min(), 3),
    'fx_rates: USD highest (about)': round(usd['rate'].max(), 3),
    'ledger_nav: funds': int(ledger_level['distinct fund_id']),
    'ledger_nav: share classes': int(ledger_level['distinct share_class']),
    'ledger_nav: dates': int(ledger_level['distinct as_of_date']),
    'REVERSED rows with related_transaction_id': int(status_links.loc['REVERSED', 'related_transaction_id filled']),
    'REVERSED rows pointing to a BOOKED record': int(link_kinds.get('REVERSED -> BOOKED', 0)),
    'BOOKED rows with related_transaction_id': int(status_links.loc['BOOKED', 'related_transaction_id filled']),
    'BOOKED rows pointing to a CANCELLED record': int(link_kinds.get('BOOKED -> CANCELLED', 0)),
    'CANCELLED rows with related_transaction_id': int(status_links.loc['CANCELLED', 'related_transaction_id filled']),
    'CANCELLED rows pointing to an ID not in transactions': int((dangling['status'] == 'CANCELLED').sum()),
    'quantity negative': len(negative_rows),
    'quantity negative on BUY rows': int((negative_rows['transaction_type'] == 'BUY').sum()),
    'quantity blank': len(blank_rows),
    'quantity blank on BUY rows': int((blank_rows['transaction_type'] == 'BUY').sum()),
    'quantity blank on SELL rows': int((blank_rows['transaction_type'] == 'SELL').sum()),
    f'trade_date after {AS_OF_DATE}': cutoff_all_rows['trade_date after the cut-off'],
    f'settlement_date after {AS_OF_DATE}': cutoff_all_rows['settlement_date after the cut-off'],
    'settlement_date before trade_date': cutoff_all_rows['settlement_date before trade_date'],
})
NOTEBOOK_ESTIMATES = {
    'pairs differing only by date format': int(pair_differs_only_by_format.sum()),
    'account_ids in positions missing from accounts': len(orphan_account_ids),
    'security_ids in positions missing from securities': len(orphan_security_ids),
}
# New numbers from sections 7 to 9 that we never worked out by hand: shown, not scored.
NEW_RESULTS = {}
for name in ['C0 raw benchmark', benchmark_name, 'C7 trade-date cut-off', 'C7s settlement-date cut-off',
             book_policy_name]:
    for reported_name in reported_tables:
        summary = step_results[name][reported_name]
        for measure in ['matched', 'different', RATE_A]:
            NEW_RESULTS[f'rebuild {name} vs {reported_name}: {measure}'] = summary[measure]
NEW_RESULTS.update({
    'transaction_count policies that fit every key': count_policies_all_keys,
    'book-only 480 supported by trades (BOOKED-only)': book_only_benchmark.get('matched', 0),
    'REVERSED originals with two REVERSED rows': len(two_reversals),
    'rebooks whose gross_amount fits the cancelled quantity': int(rebook_changed['gross_fits_target_quantity']),
    'negative quantities whose gross_amount fits the absolute value': int(fits_absolute.sum()),
    'price rows with a currency missing from fx_rates': int(prices['currency'].isin(xyz_like).sum()),
    'securities missing the preferred vendor price on a date': len(missing_vendor_dates),
    'rebuild numbers where pandas and DuckDB agree': int((rebuild_check['agree?'] == 'yes').sum()),
})

cross_check_rows = []
for check, manual_value in MANUAL_RESULTS.items():
    notebook_value = NOTEBOOK_RESULTS.get(check, 'not computed')
    cross_check_rows.append({'check': check, 'our manual result': manual_value,
                             'this notebook': notebook_value,
                             'match?': yes_no(notebook_value == manual_value)})
for check, manual_value in MANUAL_ESTIMATES.items():
    cross_check_rows.append({'check': check, 'our manual result': manual_value,
                             'this notebook': NOTEBOOK_ESTIMATES[check],
                             'match?': 'estimate only'})
for check, notebook_value in NEW_RESULTS.items():
    cross_check_rows.append({'check': check, 'our manual result': 'no manual result',
                             'this notebook': notebook_value, 'match?': 'no manual result'})

cross_check = pd.DataFrame(cross_check_rows)
cross_check.to_csv(OUTPUTS / '01_cross_check.csv', index=False)

print('match:', (cross_check['match?'] == 'yes').sum(),
      '| mismatch:', (cross_check['match?'] == 'no').sum(),
      '| estimate only:', (cross_check['match?'] == 'estimate only').sum(),
      '| no manual result:', (cross_check['match?'] == 'no manual result').sum())
display(cross_check[cross_check['match?'] == 'no'])
cross_check

**Mismatches and estimates.** In this run there is **no mismatch**: all 87 scored checks agree with our manual results. That includes the 36 new ones for sections 7 to 9: spin-off targets, mergers, corporate-action dates and cash rates, prices, FX, ledger, links, negative and blank quantities, and the cut-off counts.

- **The three estimates:**
  - Pairs that differ only by date format are 28, as we guessed.
  - 195 account_ids in positions are missing from accounts, exactly our lower bound.
  - 15 security_ids are missing from securities, well above our lower bound of 3. The guess only counted the difference in distinct IDs (6,015 minus 6,012) and ignored that the security master also contains securities nobody holds.
- **The 38 lines marked "no manual result"** are new numbers from the rebuild and the link analysis that nobody worked out by hand. They are listed so they can be checked later, not scored.

If a later run shows a mismatch, rerun the cell that produces that number and print the rows behind it. The most likely cause, on either side, is a different date normalisation or a forgotten filter (for example counting duplicate copies or not).

### 11.3 Live headline numbers

The summary box at the top was written from a run of this notebook. The cell below prints the same headline numbers live, so after any rerun you can check that the box is still true.

In [ ]:
print(f"Integrity: {(shape_check['result'] == 'PASS').sum()} of {len(shape_check)} tables match manifest.csv")
print(f"transaction_id: {NOTEBOOK_RESULTS['distinct transaction_ids']:,} distinct in "
      f"{len(transactions):,} rows; {NOTEBOOK_RESULTS['repeated transaction_ids']:,} IDs repeat")
print(f"Repeated pairs: {pair_types['exact copies']:,} exact copies, "
      f"{pair_types['differ only by date format']:,} differ only by date format, "
      f"{pair_types['real conflict (at least one column)']:,} real conflicts")
print(f"Positions vs custodian: {merge_counts['in both']:,} in both, "
      f"{merge_counts['only in positions']:,} only in positions, "
      f"{merge_counts['only in comparison_positions']:,} only in comparison_positions")
print(f"Orphans in positions: {len(orphan_account_ids):,} account_ids ({rows_with_orphan_account:,} rows), "
      f"{len(orphan_security_ids):,} security_ids ({rows_with_orphan_security:,} rows)")
print('Small tables:')
for line in suspicious_small_tables:
    print('  -', line)
print(f"Links: REVERSED -> BOOKED {link_kinds.get('REVERSED -> BOOKED', 0):,} rows on {len(reversals_per_target)} "
      f"originals ({sorted(int(count) for count in reversals_per_target.unique())} per original); BOOKED -> CANCELLED "
      f"{link_kinds.get('BOOKED -> CANCELLED', 0):,} (gross_amount fits the cancelled quantity in "
      f"{int(rebook_changed['gross_fits_target_quantity']):,} of {int(rebook_changed['rows_with_changed_quantity']):,} "
      f"changed rebooks); CANCELLED -> missing ID {len(dangling):,}")
print(f"Negative quantities: {len(negative_rows)} (all BUY: {(negative_rows['transaction_type'] == 'BUY').all()}; "
      f"gross fits the absolute value in {int(fits_absolute.sum())}); blank quantities: {len(blank_rows):,}")
for name in ['C0 raw benchmark', benchmark_name, 'C7 trade-date cut-off', 'C7s settlement-date cut-off',
             book_policy_name]:
    summary = step_results[name]['positions']
    print(f"Rebuild vs positions, {name}: {summary['matched']:,} matched, {summary['different']:,} different, "
          f"{summary['only in reported table']:,} only in positions, "
          f"{summary['not evaluable (in both)'] + summary['not evaluable (only in rebuild)']:,} not evaluable, "
          f"{summary['only in rebuild, nets to zero']} rebuilt zero keys absent; rate A {summary[RATE_A]:.2%}")
print(f"transaction_count fits every key under {count_policies_all_keys} policies: "
      f"{', '.join(count_fit.index[count_fit['transaction_count equal'] == count_fit['keys in both']])}")
print(f"Book-only 480: trades support the book on {book_only_benchmark.get('matched', 0)} under BOOKED-only")
print(f"pandas and DuckDB agree on all {len(pandas_vs_duckdb)} basic checks: "
      f"{(pandas_vs_duckdb['agree?'] == 'yes').all()} and on all {len(rebuild_check)} rebuild numbers: "
      f"{(rebuild_check['agree?'] == 'yes').all()}")
print(f"Cross-check: {(cross_check['match?'] == 'yes').sum()} match, "
      f"{(cross_check['match?'] == 'no').sum()} mismatch, "
      f"{(cross_check['match?'] == 'no manual result').sum()} new numbers with no manual result")